In [ ]:
import pandas as pd
import tqdm
import seaborn as sns
#import matplotlib.pyplot as plt
import numpy as np 
from sklearn.manifold import TSNE
from sklearn import cluster, datasets, mixture
#import matplotlib
import cmasher as cmr
#from mycolorpy import colorlist as mcp
import umap
from sklearn.manifold import TSNE
from scipy import stats
from scipy.stats import rankdata

In [ ]:
mdSAM = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')
AF2 = pd.read_pickle('../files/AF2_with_ENSG.pkl')
KeyWord_df = pd.read_csv('/Users/aelazar/Downloads/keywords_all_2023_10_27.tsv',delimiter='\t')
hTMs_uniprot = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_10_27.tsv',delimiter='\t')
hTMs_uniprot = hTMs_uniprot[~pd.isna(hTMs_uniprot.Keywords)]
#KeyWord_df = pd.read_csv('/Users/aelazar/Downloads/keywords_all_2023_10_27.tsv',delimiter='\t')

In [ ]:
KeyWord_df[KeyWord_df.Category.isin(['Biological process','Molecular function'])]

In [ ]:
IDs = set(KeyWord_df[KeyWord_df.Category.isin(['Biological process','Molecular function'])]['Keyword ID'])#,'Disease'

In [ ]:
KEYdf_counts

In [ ]:
KEY = []
for j in hTMs_uniprot[(~pd.isna(hTMs_uniprot['Keyword ID']))].index: #(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    #hTMs_uniprot.loc[0]['Gene Ontology (biological process)']
    t= [i.replace(' ','') for i in hTMs_uniprot.loc[j,'Keyword ID'].split(';')]
    if len(set.intersection(IDs,set(t)))>0:
        for key in set.intersection(IDs,set(t)):
            KEY.append({'ID':key,'disrcption':KeyWord_df[KeyWord_df['Keyword ID']==key].Name.values[0],'acc':hTMs_uniprot.loc[j,'Entry']})
KEYdf = pd.DataFrame(KEY)
KEYdf_counts = KEYdf.ID.value_counts().reset_index()

KEYdf_counts= KEYdf_counts.rename(columns={'ID': 'counts_total','index':'ID'})
KEYdf_counts= KEYdf_counts.rename(columns={'counts_total': 'ID','count':'counts_total'})

for i in KEYdf_counts.index:
    id = KEYdf_counts.loc[i,'ID']
    KEYdf_counts.loc[i,'disrcption'] =KeyWord_df[KeyWord_df['Keyword ID']==id].Name.values[0]

In [ ]:
sns.barplot(x='disrcption',y='counts_total',data=top_IDs)
plt.yscale('log')
plt.xticks(rotation=90)
plt.tight_layout()

In [ ]:
hTMs_uniprot

In [ ]:
#mdSAM[mdSAM.AAseq.str.find('VNVT')!=-1]

In [ ]:
for i in KEYdf_counts.index:
    KEYdf_counts.loc[i,'parent'] = KeyWord_df[KeyWord_df['Keyword ID']==KEYdf_counts.loc[i,'ID']].Category.values[0]
#    break

In [ ]:
hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find('KW-0675')!=-1)]

def SuperKeys(ID='KW-0675'):
    KEY = []
    for j in hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)].index: #(hTMs_uniprot.Entry.isin(mdSAM.acc))&
        #hTMs_uniprot.loc[0]['Gene Ontology (biological process)']
        t= [i.replace(' ','') for i in hTMs_uniprot.loc[j,'Keyword ID'].split(';')]
        if len(set.intersection(IDs,set(t)))>0:
            for key in set.intersection(IDs,set(t)):
                KEY.append({'ID':key,'disrcption':KeyWord_df[KeyWord_df['Keyword ID']==key].Name.values[0],'acc':hTMs_uniprot.loc[j,'Entry']})
    KEYdf = pd.DataFrame(KEY)
    KEYdf_counts = KEYdf.ID.value_counts().reset_index()
    KEYdf_counts= KEYdf_counts.rename(columns={'ID': 'counts_total','index':'ID'})
    for i in KEYdf_counts.index:
        id = KEYdf_counts.loc[i,'ID']
        KEYdf_counts.loc[i,'disrcption'] =KeyWord_df[KeyWord_df['Keyword ID']==id].Name.values[0]
    return KEYdf_counts

In [ ]:
top_IDs = KEYdf_counts[KEYdf_counts.counts_total.between(1,1500)]

In [ ]:
KEYdf_counts

In [ ]:
l =[]
for i in top_IDs.index:
    ID = top_IDs.loc[i,'ID']
    disrcption = top_IDs.loc[i,'disrcption']
    in_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))])
    not_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(~hTMs_uniprot.Entry.isin(mdSAM.acc))])
    total = not_mdSAM+in_mdSAM
    l.append({'disrcption':disrcption,'val':100*(not_mdSAM/total),'mdSAM':'-'})
    l.append({'disrcption':disrcption,'val':100*(in_mdSAM/total),'mdSAM':'+'})
top_IDs_df = pd.DataFrame(l).pivot_table(index='disrcption',columns='mdSAM',values='val')

l =[]
for i in top_IDs.index:
    ID = top_IDs.loc[i,'ID']
    disrcption = top_IDs.loc[i,'disrcption']
    in_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))])
    not_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(~hTMs_uniprot.Entry.isin(mdSAM.acc))])
    total = not_mdSAM+in_mdSAM
    l.append({'disrcption':disrcption,'val':not_mdSAM,'mdSAM':'-'})
    l.append({'disrcption':disrcption,'val':in_mdSAM,'mdSAM':'+'})
top_IDs_total = pd.DataFrame(l).pivot_table(index='disrcption',columns='mdSAM',values='val')

In [ ]:
top_IDs[top_IDs.disrcption=='Ion channel']

In [ ]:
hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find('KW-0407')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))].sample()

In [ ]:
top_IDs_total.loc['Neurogenesis'].values.sum()

In [ ]:
top_IDs_total.index.values

In [ ]:
#KFatty acid biosynthesis
KeyWord_df[KeyWord_df.Name=='Immunity']

In [ ]:
(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find('KW-0391')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))])

In [ ]:
for i in top_IDs_total.index:
    top_IDs_total.loc[i,'catagory'] = top_IDs[top_IDs['disrcption']==i].parent.values[0]
parent = top_IDs_total.pop("catagory")

In [ ]:
lut = dict(zip(parent.unique(), ['#EF6009','#08AA76','#EF1702']))
row_colors = parent.map(lut)


In [ ]:
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#B96071','#679AAB','#7D8FA1'])#"#18827E","#B34F91"])#"#"

In [ ]:
def Name2int(name='Acyltransferase'):
    return int(KeyWord_df[KeyWord_df.Name==name]['Keyword ID'].values[0].split('-')[-1])

In [ ]:
top_IDs_df_reset = top_IDs_df.reset_index()
top_IDs_total_reset = top_IDs_total.reset_index()
top_IDs_total_reset

In [ ]:
mapping  = dict(KeyWord_df.loc[:,['Name','Category']].values)
top_IDs_total
top_IDs_total['Category'] = top_IDs_total.index.map(mapping)
top_IDs_total.Category.value_counts()

In [ ]:
df_total = top_IDs_total.reset_index()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections import register_projection
from matplotlib.projections.polar import PolarAxes
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D


def radar_factory(num_vars, frame='circle'):
    """
    Create a radar chart with `num_vars` Axes.

    This function creates a RadarAxes projection and registers it.

    Parameters
    ----------
    num_vars : int
        Number of variables for radar chart.
    frame : {'circle', 'polygon'}
        Shape of frame surrounding Axes.

    """
    # calculate evenly-spaced axis angles
    theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)

    class RadarTransform(PolarAxes.PolarTransform):

        def transform_path_non_affine(self, path):
            # Paths with non-unit interpolation steps correspond to gridlines,
            # in which case we force interpolation (to defeat PolarTransform's
            # autoconversion to circular arcs).
            if path._interpolation_steps > 1:
                path = path.interpolated(num_vars)
            return Path(self.transform(path.vertices), path.codes)

    class RadarAxes(PolarAxes):

        name = 'radar'
        PolarTransform = RadarTransform

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            # rotate plot such that the first axis is at the top
            self.set_theta_zero_location('N')

        def fill(self, *args, closed=True, **kwargs):
            """Override fill so that line is closed by default"""
            return super().fill(closed=closed, *args, **kwargs)

        def plot(self, *args, **kwargs):
            """Override plot so that line is closed by default"""
            lines = super().plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)

        def _close_line(self, line):
            x, y = line.get_data()
            # FIXME: markers at x[0], y[0] get doubled-up
            if x[0] != x[-1]:
                x = np.append(x, x[0])
                y = np.append(y, y[0])
                line.set_data(x, y)

        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)

        def _gen_axes_patch(self):
            # The Axes patch must be centered at (0.5, 0.5) and of radius 0.5
            # in axes coordinates.
            if frame == 'circle':
                return Circle((0.5, 0.5), 0.5)
            elif frame == 'polygon':
                return RegularPolygon((0.5, 0.5), num_vars,
                                      radius=.5, edgecolor="k")
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

        def _gen_axes_spines(self):
            if frame == 'circle':
                return super()._gen_axes_spines()
            elif frame == 'polygon':
                # spine_type must be 'left'/'right'/'top'/'bottom'/'circle'.
                spine = Spine(axes=self,
                              spine_type='circle',
                              path=Path.unit_regular_polygon(num_vars))
                # unit_regular_polygon gives a polygon of radius 1 centered at
                # (0, 0) but we want a polygon of radius 0.5 centered at (0.5,
                # 0.5) in axes coordinates.
                spine.set_transform(Affine2D().scale(.5).translate(.5, .5)
                                    + self.transAxes)
                return {'polar': spine}
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

    register_projection(RadarAxes)
    return theta

In [ ]:
df_total.sort_values(by='+').tail(20)

In [ ]:
df_total.loc[df_total[df_total.Category=='Biological process'].index,'color'] = '#13609A'
df_total.loc[df_total[df_total.Category=='Molecular function'].index,'color'] = '#3D3E3C'
df_total.loc[df_total[df_total.Category=='Disease'].index,'color'] = '#766094'
hand_pick = ['Cell adhesion','G-protein coupled receptor','Differentiation','Immunity','Ion transport','Kinase','Epilepsy']
df_total[df_total.disrcption.isin(hand_pick)]

In [ ]:
df_total.loc[df_total[df_total.disrcption.isin(hand_pick)].index,'ex']= 0.2
df_total.loc[df_total[~df_total.disrcption.isin(hand_pick)].index,'ex']= 0

In [ ]:
def autopct_format(values):
    def my_format(pct):
        total = sum(values)
        val = int(round(pct*total/100.0))
        return '{v:d}'.format(v=val)
    return my_format

# create data
size_of_groups=df_total.sort_values(by='+')['+']

# Create a pieplot
plt.pie(size_of_groups,colors=list(df_total.sort_values(by='+').color.values),autopct = autopct_format(size_of_groups),
       pctdistance=1.25, labeldistance=.6,explode=list(df_total.sort_values(by='+').ex.values))


# add a circle at the center to transform it in a donut chart
my_circle=plt.Circle( (0,0), 0.7, color='w')
p=plt.gcf()
p.gca().add_artist(my_circle)

plt.savefig('donut_v4.svg',bbox_inches='tight',transparent=True)

In [ ]:
df_total[df_total.disrcption.isin(hand_pick)]

In [ ]:
N = 9
theta = radar_factory(N, frame='polygon')
fig, axs = plt.subplots(figsize=(5, 5), nrows=1, ncols=1,
                        subplot_kw=dict(projection='radar'))
fig.subplots_adjust(wspace=0.25, hspace=0.20, top=0.85, bottom=0.05)
ax.plot(theta, df_total[df_total.disrcption.isin(hand_pick)]['+'], color=color)

In [ ]:
def hypergeom(KeyWord):
    total_runners =  len(hTMs_uniprot) 
    number_team_A = len(hTMs_uniprot[hTMs_uniprot.Entry.isin(mdSAM.acc)]) # number mdSAM proteins 
    number_winners = df_total[df_total.disrcption==KeyWord]['+'].values[0] + df_total[df_total.disrcption==KeyWord]['-'].values[0]
    number_win_from_A = df_total[df_total.disrcption==KeyWord]['+'].values[0]
    print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 


In [ ]:
def fdr(p_vals):
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

In [ ]:
hypergeom('Hydrogen ion transport')

In [ ]:
l =[]
for i in df_total.index:
    Key = df_total.loc[i,'disrcption']
    Category = df_total.loc[i,'Category']
    number_win_from_A,expected,p  = hypergeom(Key)
    l.append({'KeyWord':Key,'Category':Category,'obs':number_win_from_A,'expected':expected,'pval':p})
df_volcano = pd.DataFrame(l)
df_volcano['obs_exp'] = df_volcano['obs']- df_volcano['expected']

In [ ]:

df_volcano = df_volcano.drop(index=df_volcano[df_volcano.pval==0].index)
ranked_p_values = rankdata(df_volcano['pval'])
fdr = df_volcano['pval'] * len(df_volcano['pval']) / ranked_p_values
fdr[fdr > 1] = 1
df_volcano['fdr'] = fdr
df_volcano['fdr_log'] = -1*np.log10(df_volcano['fdr'])

In [ ]:
df_volcano.to_excel('UniProt_Keyword_enrich.xlsx')

In [ ]:
from adjustText import adjust_text


In [ ]:
import cmasher as cmr
from random import shuffle
colors = cmr.take_cmap_colors('tab20c', 30, return_fmt='hex')
#print(colors)#(colors))
shuffle(colors)
print(colors)

In [ ]:
import matplotlib as mpl
from matplotlib import cm
norm = mpl.colors.Normalize(vmin=0, vmax=25)
class MidpointNormalize(colors.Normalize):
    def __init__(self, vmin=None, vmax=None, vcenter=None, clip=False):
        self.vcenter = vcenter
        super().__init__(vmin, vmax, clip)

    def __call__(self, value, clip=None):
        # I'm ignoring masked values and all kinds of edge cases to make a
        # simple example...
        # Note also that we must extrapolate beyond vmin/vmax
        x, y = [self.vmin, self.vcenter, self.vmax], [0, 0.5, 1.]
        return np.ma.masked_array(np.interp(value, x, y,
                                            left=-np.inf, right=np.inf))

    def inverse(self, value):
        y, x = [self.vmin, self.vcenter, self.vmax], [0, 0.5, 1]
        return np.interp(value, x, y, left=-np.inf, right=np.inf)


midnorm = MidpointNormalize(vmin=0., vcenter=2, vmax=25)
divnorm = colors.TwoSlopeNorm(vcenter=2)

In [ ]:
e = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D1D0","#043278"])
# Create a ScalarMappable object
scalar_map = cm.ScalarMappable(norm=divnorm, cmap=e)
scalar_map.set_array([])  # Set an empty array to make the ScalarMappable work

# Create a custom colormap
custom_cmap = scalar_map.cmap

sns.scatterplot(x='rank',y='FC',data=df,color=color,s=60,hue='FDR',edgecolor=None,palette=custom_cmap)


In [ ]:
df

In [ ]:
for pro,color in zip(['Biological process', 'Disease', 'Molecular function'],['#125F99','#6E578E','#3D3E3C']):
    l = []
    for i in enumerate(df_volcano[(df_volcano.Category==pro)].sort_values(by='obs_exp').index):
        fc = df_volcano.loc[i[1],'obs_exp']
        fdr = df_volcano.loc[i[1],'fdr_log']
        KeyWord = df_volcano.loc[i[1],'KeyWord']
        l.append({'FC':fc,'FDR':fdr,'rank':i[0],'KeyWord':KeyWord})
    fig, ax = plt.subplots(figsize=(4,4))
    df  = pd.DataFrame(l)
    sns.scatterplot(x='rank',y='FC',data=df[df.FDR>2],color=color,size='FDR',size_norm=norm,sizes=(30,250),edgecolor=None)
    sns.scatterplot(x='rank',y='FC',data=df[df.FDR<2],color='#A6A6A6',size='FDR',size_norm=norm,sizes=(30,250),edgecolor=None)
    ax.set_xlim([-10,len(df)+20])
    texts = []
    for i in df[(df.FDR>2)].index:
        texts.append(ax.text(df.loc[i,'rank'],df.loc[i,'FC'],
            '%s'%df.loc[i,'KeyWord'],
                          fontsize=15))
        
    adjust_text(texts, autoalign='y',ax=ax,
                arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
    plt.legend()
    ax.grid(zorder=0)
    ax.axhline(0,ls='--',color='k',lw=0.5)
    ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
    ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
    ax.set_title('%s'%pro,fontsize=24)
    ax.set_xlabel('Rank',fontsize=16)
    ax.set_ylabel('Obs-Exp',fontsize=16)
    plt.savefig('volcano_%s.svg'%pro.replace(' ','_'),bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='obs_exp',y='fdr_log',data=df_volcano,hue='Category',ax=ax,edgecolor=None,hue_order=['Biological process','Molecular function','Disease']
                ,palette=['#13609A','#3D3E3C','#766094'])
texts = []
for i in df_volcano[(df_volcano.fdr_log.between(3,35))&((df_volcano.obs_exp>11)| (df_volcano.obs_exp<-11))].index:
    texts.append(plt.text(df_volcano.loc[i,'obs_exp'],df_volcano.loc[i,'fdr_log'],
            '%s'%df_volcano.loc[i,'KeyWord'],
                          fontsize=10))
    
adjust_text(texts, autoalign='xy',
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)

In [ ]:
df_volcano.Category.value_counts()

In [ ]:
f, (ax_top, ax_bottom) = plt.subplots(ncols=1, nrows=2, sharex=True, gridspec_kw={'hspace':0.05},figsize=(5, 5))

sns.scatterplot(x='obs_exp',y='fdr_log',data=df_volcano[(df_volcano.fdr_log.between(3,35))&((df_volcano.obs_exp>11)| (df_volcano.obs_exp<-11))],hue='Category',ax=ax_top,edgecolor=None,hue_order=['Biological process','Molecular function','Disease']
                ,palette=['#13609A','#3D3E3C','#766094'],s=100)
sns.scatterplot(x='obs_exp',y='fdr_log',data=df_volcano[(df_volcano.fdr_log.between(3,35))&((df_volcano.obs_exp>11)| (df_volcano.obs_exp<-11))],hue='Category',ax=ax_bottom,edgecolor=None,hue_order=['Biological process','Molecular function','Disease']
                ,palette=['#13609A','#3D3E3C','#766094'],s=100)
# sns.scatterplot(x='obs_exp',y='fdr_log',data=df_volcano[(df_volcano.fdr_log.between(0,3))&((df_volcano.obs_exp<11)| (df_volcano.obs_exp<-11))],hue='Category',ax=ax_top,edgecolor=None,hue_order=['Biological process','Molecular function','Disease']
#                 ,palette=['#13609A','#3D3E3C','#766094'],alpha=.1,zorder=4)
sns.scatterplot(x='obs_exp',y='fdr_log',data=df_volcano[(df_volcano.fdr_log.between(0,3))&((df_volcano.obs_exp<11)| (df_volcano.obs_exp<-11))],hue='Category',ax=ax_bottom,edgecolor=None,hue_order=['Biological process','Molecular function','Disease']
                ,palette=['#13609A','#3D3E3C','#766094'],alpha=.1,zorder=4)
ax_top.set_ylim(bottom=7)   # those limits are fake
ax_bottom.set_ylim(-.5,7)
ax_bottom.set_xlim(-80,70)

ax_bottom.axvline(-11,color='k',ls='--')
ax_bottom.axvline(11,color='k',ls='--')
ax_bottom.axhline(3,color='k',ls='--')
ax_bottom.set_xlim(-80,80)

ax_bottom.set_xlim(-80,80)

ax_top.set_xlim(-80,80)
ax_bottom.set_ylabel('')
ax_bottom.set_xlabel('')
ax_top.set_xlabel('')
ax_top.set_ylabel('')
ax
# ax_top.grid(zorder=0)
# ax_bottom.grid(zorder=0)
ax_bottom.xaxis.set_major_locator(plt.MaxNLocator(4))
ax_bottom.yaxis.set_major_locator(plt.MaxNLocator(2))
ax_bottom.set_yticklabels(ax_bottom.get_yticklabels(), rotation=0,fontsize=16)
ax_bottom.set_xticklabels(ax_bottom.get_xticklabels(), rotation=0,fontsize=16)
ax_top.yaxis.set_major_locator(plt.MaxNLocator(2))
ax_top.set_yticklabels(ax_top.get_yticklabels(), rotation=0,fontsize=16)

texts = []
for i in df_volcano[(df_volcano.fdr_log.between(3,7))&((df_volcano.obs_exp>11)| (df_volcano.obs_exp<-11))].index:
    texts.append(plt.text(df_volcano.loc[i,'obs_exp'],df_volcano.loc[i,'fdr_log'],
            '%s'%df_volcano.loc[i,'KeyWord'],
                          fontsize=14))

adjust_text(texts, autoalign='xy',ax=ax_bottom,
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=10)

# texts = []
# for i in df_volcano[(df_volcano.fdr_log.between(7,35))].index:
#     texts.append(plt.text(df_volcano.loc[i,'obs_exp'],df_volcano.loc[i,'fdr_log'],
#             '%s'%df_volcano.loc[i,'KeyWord'],
#                           fontsize=14))

# adjust_text(texts, autoalign='xy',ax=ax_top,force_objects=(0., 0.2),
#             arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)

ax = ax_top
d = .01  # how big to make the diagonal lines in axes coordinates
# arguments to pass to plot, just so we don't keep repeating them
kwargs = dict(transform=ax.transAxes, color='k', clip_on=False)
ax.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal

ax2 = ax_bottom
kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal
sns.despine(ax=ax2)
sns.despine(ax=ax, bottom=True)
#remove one of the legend
ax_bottom.legend_.remove()
plt.savefig('c.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:

plt.ylim([0,6])

In [ ]:
plt.rcParams["font.family"] = "Arial"
for pro,color in zip(['Biological process', 'Disease', 'Molecular function'],['#125F99','#6E578E','#3D3E3C']):
    l = []
    for i in enumerate(df_volcano[(df_volcano.Category==pro)].sort_values(by='obs_exp').index):
        fc = df_volcano.loc[i[1],'obs_exp']
        fdr = df_volcano.loc[i[1],'fdr_log']
        KeyWord = df_volcano.loc[i[1],'KeyWord']
        l.append({'FC':fc,'FDR':fdr,'rank':i[0],'KeyWord':KeyWord})
    fig, ax = plt.subplots(figsize=(4,4))
    df  = pd.DataFrame(l)
    sns.scatterplot(x='FC',y='rank',data=df[df.FDR>2],color=color,size='FDR',size_norm=norm,sizes=(30,250),zorder=3,edgecolor=None,alpha=.95)
    sns.scatterplot(x='FC',y='rank',data=df[df.FDR<2],color='#A6A6A6',size='FDR',size_norm=norm,sizes=(30,250),edgecolor=None)
    #ax.set_yticks(ax.get_yticks().tolist()); ax.set_xticks(ax.get_xticks().tolist())
    #ax.set_yticks(ax.get_yticks().tolist()); ax.set_xticks(ax.get_xticks().tolist())
    ax.set_ylim([-20,len(df)+20])
    ax.set_xlim([df[df.FDR>2].FC.min()-20,df[df.FDR>2].FC.max()+20])

    texts = []
    for i in df[(df.FDR>2)].index:
        texts.append(ax.text(df.loc[i,'FC'],df.loc[i,'rank'],
            '%s'%df.loc[i,'KeyWord'],
                          fontsize=12))
        
    adjust_text(texts, autoalign='y',ax=ax,
                arrowprops=dict(arrowstyle="-", color='gray', lw=1),zorder=0)
    ax.grid(zorder=0)
    ax.axvline(0,ls='--',color='k',lw=0.5)

    ax.set_title('%s'%pro,fontsize=24)
    ax.set_ylabel('Rank',fontsize=16)
    ax.set_xlabel('Obs-Exp',fontsize=16)
    ax.locator_params(axis='x', nbins=5)
    ax.locator_params(axis='y', nbins=5)
    ax.xaxis.set_tick_params(labelsize=16)
    ax.yaxis.set_tick_params(labelsize=16)
    #ax.legend([])
    plt.savefig('volcano_%s.svg'%pro.replace(' ','_'),bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
for i in zip(['Biological process', 'Disease', 'Molecular function'],['#3B7FB0','#6E578E','#3D3E3C']):
    fig, ax = plt.subplots(figsize=(7,7))
    
    sns.scatterplot(x='FC',y='fdr_log',data=df_volcano[(df_volcano.Category==i[0])&(df_volcano.fdr_log>-1*np.log10(0.01))&(~df_volcano.FC.between(-.5,.5))],
                    edgecolor=None,color=i[1],s=200)
    sns.scatterplot(x='FC',y='fdr_log',data=df_volcano[(df_volcano.Category==i[0])&
                (df_volcano.fdr_log<-1*np.log10(0.01))],ax=ax,edgecolor=None,color='#D0D1D0',s=100)
    ax.set_xlim([-8,8])
    texts = []
    for j in df_volcano[(df_volcano.Category==i[0])&
                (df_volcano.fdr_log>-1*np.log10(0.01))&(~df_volcano.FC.between(-.5,.5))].index:
        texts.append(ax.text(df_volcano.loc[j,'FC'],df_volcano.loc[j,'fdr_log'],
            '%s'%df_volcano.loc[j,'KeyWord'],
                          fontsize=22))
    print(texts)
    adjust_text(texts, autoalign='y',ax=ax,
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
    #ax.set_xlim([-60,60])
    plt.axhline(2,ls='--',c='gray')
    plt.axvline(-.5,ls='--',c='gray')
    plt.axvline(.5,ls='--',c='gray')
    ax.grid(zorder=0)
    ax.set_ylabel('FDR(-log$_{10}$)',fontsize=16)
    ax.set_xlabel('FC(log$_{2}$)',fontsize=16)
    ax.set_title('%s'%i[0],fontsize=24)
    ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
    ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
    plt.show()
#    plt.savefig('volcano_%s.svg'%i[0].replace(' ','_'),bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
'Biological process'

In [ ]:
df_volcano['obs'] = df_volcano['obs']+.5

In [ ]:
df_volcano['FC']=np.log2(df_volcano['obs']/df_volcano['expected'])

In [ ]:
df_volcano[df_volcano.KeyWord.str.find('inase')!=-1]

In [ ]:
top_IDs_df_reset

In [ ]:
l =[]
for i in top_IDs_df_reset.index:
    l.append({'val':top_IDs_df_reset.loc[i,'+'],'word':top_IDs_df_reset.loc[i,'disrcption'],'mdSAM':'+'})
    l.append({'val':top_IDs_df_reset.loc[i,'-'],'word':top_IDs_df_reset.loc[i,'disrcption'],'mdSAM':'-'})
ll = pd.DataFrame(l).pivot_table(index='mdSAM',columns='word',values='val')

l =[]
for i in top_IDs_total_reset.index:
    l.append({'val':top_IDs_total_reset.loc[i,'+'],'word':top_IDs_total_reset.loc[i,'disrcption'],'mdSAM':'+'})
    l.append({'val':top_IDs_total_reset.loc[i,'-'],'word':top_IDs_total_reset.loc[i,'disrcption'],'mdSAM':'-'})
ll_total = pd.DataFrame(l).pivot_table(index='mdSAM',columns='word',values='val')

In [ ]:
lut = dict(zip(parent.unique(), ['#3B7FB0','#6E578E','#3D3E3C']))
row_colors = parent.map(lut)


In [ ]:

g  = sns.clustermap(ll,cmap='binary',annot=ll_total,fmt='.4g',annot_kws={"size": 20,'rotation':90},
           col_colors=row_colors,figsize=(100, 5),col_cluster=True,row_cluster=False,dendrogram_ratio=.1,
center=75,cbar_pos=(0.99, 0.5, 0.03, 0.05),)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize = 24)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize = 24)
g.ax_heatmap.set_ylabel('mdSAM', fontsize = 24)
plt.savefig('KEyWords_heatmap.svg',bbox_inches='tight',transparent=True)

In [ ]:
Top_KeyWord = []
for i in ll_total.columns:
    if (ll_total.loc[:,i].sum())>=30:
        Top_KeyWord.append(i)
    #break

In [ ]:
len(Top_KeyWord)

In [ ]:

g  = sns.clustermap(ll_total.loc[:,Top_KeyWord],cmap='binary',annot=ll_total.loc[:,Top_KeyWord],fmt='.4g',annot_kws={"size": 12,'rotation':90},
           col_colors=row_colors,figsize=(105, 20),col_cluster=True,row_cluster=False,dendrogram_ratio=.1,
center=75,cbar_pos=(0.99, 0.5, 0.03, 0.05),)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize = 12)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize = 12)
g.ax_heatmap.set_ylabel('mdSAM', fontsize = 12)
#plt.savefig('KEyWords_heatmap.svg',bbox_inches='tight',transparent=True)

In [ ]:
# sns.clustermap(top_IDs_df.sort_values(by='+'),cmap='binary',col_cluster=False,row_cluster=False,row_colors=row_colors,
#             annot=top_IDs_total.loc[top_IDs_df.sort_values(by='+').index],fmt='.4g',figsize=(4,14),lw=0,center=75,cbar_pos=(0.9, 0.5, 0.03, 0.05),cbar_kws={'label':'%'},linecolor='w')

# plt.savefig('KEyWords_heatmap.svg',bbox_inches='tight',transparent=True)

In [ ]:
l =[]
for i in top_IDs_df_reset.index:
    l.append({'val':top_IDs_df_reset.loc[i,'+'],'word':top_IDs_df_reset.loc[i,'disrcption'],'mdSAM':'+'})
    l.append({'val':top_IDs_df_reset.loc[i,'-'],'word':top_IDs_df_reset.loc[i,'disrcption'],'mdSAM':'-'})
ll = pd.DataFrame(l).pivot_table(index='mdSAM',columns='word',values='val')

l =[]
for i in top_IDs_total_reset.index:
    l.append({'val':top_IDs_total_reset.loc[i,'+'],'word':top_IDs_total_reset.loc[i,'disrcption'],'mdSAM':'+'})
    l.append({'val':top_IDs_total_reset.loc[i,'-'],'word':top_IDs_total_reset.loc[i,'disrcption'],'mdSAM':'-'})
ll_total = pd.DataFrame(l).pivot_table(index='mdSAM',columns='word',values='val')

In [ ]:
ll = pd.DataFrame(l)
ll[ll.word.isin(Top_KeyWord)].groupby(['word','mdSAM']).sum()

In [ ]:
len(set(mdSAM.acc))
hTMs = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')
ratio_base = len(set(mdSAM.acc))/len(set(hTMs.acc))
z = []
for word in Top_KeyWord:
    p = (ll[(ll.word==word)&(ll.mdSAM=='+')].val.values[0])
    n = (ll[(ll.word==word)&(ll.mdSAM=='-')].val.values[0])
    ratio = (p/n)/ratio_base
    ratio_p = p/(p+n)
    ratio_n = 1- ratio_p
    z.append({'ratio_normalized':ratio,'+_ratio':ratio_p,
              '-_ratio':ratio_n
              ,'+':p,'-':n,'word':word,'cat':KeyWord_df[KeyWord_df.Name==word]['Category'].values[0]})
    #/ll[ll.word==word)&(ll.mdSAM=='+')].val)

In [ ]:
zz  = pd.DataFrame(z)
zz

In [ ]:

plus = []
minus = []
TM = [] 
species = []
orderd_spe = zz[zz.cat=='Disease'].sort_values(by='+_ratio')['word'].to_list()
for i in orderd_spe:
    species.append(i)
    plus.append(zz[zz.word==i]['+_ratio'].values[0])
    minus.append(zz[zz.word==i]['-_ratio'].values[0])
species.append('Base ratio')
plus.append(ratio_base)
minus.append(1-ratio_base)
species= tuple(species)
weight_counts = {
    "+": np.array(plus),
    "-": np.array(minus),
    
}
width = 1

fig, ax = plt.subplots(figsize=(3,3))
bottom = np.zeros(len(orderd_spe)+1)
colors=['#3C4676','#B4B5B4']
c  = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
for i in enumerate(orderd_spe):
    ax.text(i[0]-.25,0.1,'%d'%zz[zz.word==i[1]]['+'].values[0],rotation=90,c='white')
for i in enumerate(orderd_spe):
    ax.text(i[0]-.25,0.5,'%d'%zz[zz.word==i[1]]['-'].values[0],rotation=90)
ax.set_title("Disease")
plt.xlabel("Keyword")
plt.xlabel("Fraction")
ax.legend(loc="upper right")
plt.xticks(rotation=90)
plt.savefig('Disease_top_Keyword_bar.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:

plus = []
minus = []
TM = [] 
species = []
orderd_spe = zz[zz.cat=='Molecular function'].sort_values(by='+_ratio')['word'].to_list()
for i in orderd_spe:
    species.append(i)
    plus.append(zz[zz.word==i]['+_ratio'].values[0])
    minus.append(zz[zz.word==i]['-_ratio'].values[0])
species.append('Base ratio')
plus.append(ratio_base)
minus.append(1-ratio_base)
species= tuple(species)
weight_counts = {
    "+": np.array(plus),
    "-": np.array(minus),
    
}
width = 1

fig, ax = plt.subplots(figsize=(6,3))
bottom = np.zeros(len(orderd_spe)+1)
colors=['#3C4676','#B4B5B4']
c  = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
for i in enumerate(orderd_spe):
    ax.text(i[0]-.25,0.1,'%d'%zz[zz.word==i[1]]['+'].values[0],rotation=90,c='white')
for i in enumerate(orderd_spe):
    ax.text(i[0]-.25,0.6,'%d'%zz[zz.word==i[1]]['-'].values[0],rotation=90)
ax.set_title("Molecular function")
plt.xlabel("Keyword")
plt.xlabel("Fraction")
ax.legend(loc="upper right")
plt.xticks(rotation=90)
plt.savefig('Molecular_top_Keyword_bar.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:

plus = []
minus = []
TM = [] 
species = []
orderd_spe = zz[zz.cat=='Biological process'].sort_values(by='+_ratio')['word'].to_list()
for i in orderd_spe:
    species.append(i)
    plus.append(zz[zz.word==i]['+_ratio'].values[0])
    minus.append(zz[zz.word==i]['-_ratio'].values[0])
species.append('Base ratio')
plus.append(ratio_base)
minus.append(1-ratio_base)
species= tuple(species)
weight_counts = {
    "+": np.array(plus),
    "-": np.array(minus),
    
}
width = 1

fig, ax = plt.subplots(figsize=(10,3))
bottom = np.zeros(len(orderd_spe)+1)
colors=['#3C4676','#B4B5B4']
c  = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c],linewidth=1)
    bottom += weight_count
    c = c+1
for i in enumerate(orderd_spe):
    ax.text(i[0]-.26,0.1,'%d'%zz[zz.word==i[1]]['+'].values[0],rotation=90,c='white')
for i in enumerate(orderd_spe):
    ax.text(i[0]-.26,0.6,'%d'%zz[zz.word==i[1]]['-'].values[0],rotation=90)
ax.set_title("Biological process")
plt.xlim([-1,len(orderd_spe)+1])
plt.xlabel("Keyword")
plt.xlabel("Fraction")
ax.legend(loc="upper right")
plt.xticks(rotation=90)
plt.savefig('Biological_top_Keyword_bar.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
for i in zz.index:
    zz.loc[i,'comb'] = '%s(%d,%d)'%(zz.loc[i,'word'],zz.loc[i,'+'],zz.loc[i,'-'])
set(zz.cat)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections import register_projection
from matplotlib.projections.polar import PolarAxes
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D


def radar_factory(num_vars, frame='circle'):
    """
    Create a radar chart with `num_vars` axes.

    This function creates a RadarAxes projection and registers it.

    Parameters
    ----------
    num_vars : int
        Number of variables for radar chart.
    frame : {'circle', 'polygon'}
        Shape of frame surrounding axes.

    """
    # calculate evenly-spaced axis angles
    theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)

    class RadarTransform(PolarAxes.PolarTransform):

        def transform_path_non_affine(self, path):
            # Paths with non-unit interpolation steps correspond to gridlines,
            # in which case we force interpolation (to defeat PolarTransform's
            # autoconversion to circular arcs).
            if path._interpolation_steps > 1:
                path = path.interpolated(num_vars)
            return Path(self.transform(path.vertices), path.codes)

    class RadarAxes(PolarAxes):

        name = 'radar'
        PolarTransform = RadarTransform

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            # rotate plot such that the first axis is at the top
            self.set_theta_zero_location('N')

        def fill(self, *args, closed=True, **kwargs):
            """Override fill so that line is closed by default"""
            return super().fill(closed=closed, *args, **kwargs)

        def plot(self, *args, **kwargs):
            """Override plot so that line is closed by default"""
            lines = super().plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)

        def _close_line(self, line):
            x, y = line.get_data()
            # FIXME: markers at x[0], y[0] get doubled-up
            if x[0] != x[-1]:
                x = np.append(x, x[0])
                y = np.append(y, y[0])
                line.set_data(x, y)

        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)

        def _gen_axes_patch(self):
            # The Axes patch must be centered at (0.5, 0.5) and of radius 0.5
            # in axes coordinates.
            if frame == 'circle':
                return Circle((0.5, 0.5), 0.5)
            elif frame == 'polygon':
                return RegularPolygon((0.5, 0.5), num_vars,
                                      radius=.5, edgecolor="k")
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

        def _gen_axes_spines(self):
            if frame == 'circle':
                return super()._gen_axes_spines()
            elif frame == 'polygon':
                # spine_type must be 'left'/'right'/'top'/'bottom'/'circle'.
                spine = Spine(axes=self,
                              spine_type='circle',
                              path=Path.unit_regular_polygon(num_vars))
                # unit_regular_polygon gives a polygon of radius 1 centered at
                # (0, 0) but we want a polygon of radius 0.5 centered at (0.5,
                # 0.5) in axes coordinates.
                spine.set_transform(Affine2D().scale(.5).translate(.5, .5)
                                    + self.transAxes)
                return {'polar': spine}
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

    register_projection(RadarAxes)
    return theta



In [ ]:
theta = radar_factory(len(zz[zz.cat=='Biological process']), frame='polygon')
fig, ax = plt.subplots(figsize=(7,7),subplot_kw=dict(projection='radar'))
ax.plot(theta, zz[zz.cat=='Biological process'].sort_values(by='+_ratio')['+_ratio'], color='#C03830',lw=5,zorder=2)
ax.plot(theta, zz[zz.cat=='Biological process'].sort_values(by='+_ratio')['-_ratio'], color='#B4B5B4',lw=5,zorder=2)
ax.set_varlabels(zz[zz.cat=='Biological process'].sort_values(by='+_ratio')['comb'])
ax.plot(np.linspace(0, 2*np.pi, 100), [ratio_base] * len(np.linspace(0, 2*np.pi, 100)), c='#D98883',zorder=0,lw=2,ls='--')
plt.savefig('Biological_top_Keyword_spider.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
theta

In [ ]:
for name in set(zz.cat):
    l = zz[zz.cat==name].sort_values(by='+_ratio')['comb'].to_list()
    ll =['' for i in range(len(l))]
    theta = radar_factory(len(zz[zz.cat==name]), frame='polygon')
    fig, ax = plt.subplots(figsize=(7,7),subplot_kw=dict(projection='radar'))
    ax.plot(theta, zz[zz.cat==name].sort_values(by='+_ratio')['+_ratio'], color='#C03830',lw=5,zorder=2)
    ax.plot(theta, zz[zz.cat==name].sort_values(by='+_ratio')['-_ratio'], color='#B4B5B4',lw=5,zorder=2)
    #ax.set_varlabels(zz[zz.cat==name].sort_values(by='+_ratio')['comb'])
    ax.set_varlabels(ll)
    c = .4
    for i in enumerate(theta):
        ax.text(i[1],c,'%s'%l[i[0]],rotation=90-i[1],rotation_mode='anchor',
              transform_rotates_text=1,fontsize=18)
        c = c+.01
    ax.plot(np.linspace(0, 2*np.pi, 100), [ratio_base] * len(np.linspace(0, 2*np.pi, 100)), c='#333E69',zorder=0,lw=2,ls='--')
    plt.xticks(fontsize=10
              )
    plt.savefig('%s_top_Keyword_spider.svg'%name.replace(' ',''),bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
import holoviews as hv
from holoviews import opts, dim
from bokeh.sampledata.airport_routes import routes, airports

hv.extension('bokeh')

In [ ]:
# Count the routes between Airports
route_counts = routes.groupby(['SourceID', 'DestinationID']).Stops.count().reset_index()
nodes = hv.Dataset(airports, 'AirportID', 'City')
chord = hv.Chord((route_counts, nodes), ['SourceID', 'DestinationID'], ['Stops'])
#chord = hv.Chord(route_counts, ['SourceID', 'DestinationID'], ['Stops'])
# Select the 20 busiest airports
busiest = list(routes.groupby('SourceID').count().sort_values('Stops').iloc[-20:].index.values)
busiest_airports = chord.select(AirportID=busiest, selection_mode='nodes')

In [ ]:
top_IDs_total.loc['Electron transport']['+']=1 # pseduocount

In [ ]:
w = top_IDs_total/stats.chi2_contingency(top_IDs_total)[-1]#.sort_values(by='+')
w.sort_values(by='+').tail(50)

In [ ]:
from bokeh.io import export_svgs

def export_svg(obj, filename):
    plot_state = hv.renderer('bokeh').get_plot(obj).state
    plot_state.output_backend = 'svg'
    export_svgs(plot_state, filename=filename)

In [ ]:
mapping  = dict(KeyWord_df.loc[:,['Name','Category']].values)
top_IDs_total
top_IDs_total['Category'] = top_IDs_total.index.map(mapping)
top_IDs_total

In [ ]:
top_IDs_total

In [ ]:
top_IDs_total.loc[top_IDs_df.sort_values(by='+').index].head(5).sort_values(by='Category')

In [ ]:
l = []
for i in top_IDs_total.loc[top_IDs_df.sort_values(by='+').index].tail(11).sort_values(by='Category').index:
    name = i
    for j in ['+']:
        l.append(['%s'%i,'+',int(top_IDs_total.loc[i]['%s'%j])])
sankey = hv.Sankey(l)
sankey.opts(label_position='inner', width=250, height=500, cmap='tab20b'
            ,edge_line_width=0,label_text_font_size='13pt',node_line_width=0)
#export_svg(sankey, 'mdSAM_low.svg')
#sankey = hv.Sankey(l,edge_line_width=0,label_text_font_size='12pt',node_line_width=0,label_position='inner', width=600, height=500)
# sankey.opts(width=250, height=750)

#hv.save(sankey, 'penguin_plot.png', fmt='png')
# l = []
for i in top_IDs_total.loc[top_IDs_df.sort_values(by='+').index].head(5).sort_values(by='Category').index:
    name = i
    for j in ['+']:
        l.append(['%s'%i,'+',int(top_IDs_total.loc[i]['%s'%j])])
sankey = hv.Sankey(l)
sankey.opts(label_position='inner', width=300, height=700, cmap='tab20b'
            ,edge_line_width=0,label_text_font_size='10pt',node_line_width=0)
#sankey = hv.Sankey(l,edge_line_width=0,label_text_font_size='12pt',node_line_width=0,label_position='inner', width=600, height=500)
export_svg(sankey, 'mdSAM_all_left2right.svg')


In [ ]:
top_IDs_total[top_IDs_total.Category=='Disease']

In [ ]:
ind_tail = top_IDs_total.loc[top_IDs_df.sort_values(by='+').index].tail(11).sort_values(by='Category').index
ind_head = top_IDs_total.loc[top_IDs_df.sort_values(by='+').index].head(5).sort_values(by='Category').index
flat_list = [item for sublist in [ind_tail,ind_head] for item in sublist]
top_IDs_total.loc[flat_list,:].sort_values(by='Category')

In [ ]:
top_IDs_total.to_pickle('top_IDs_total.pkl')

In [ ]:
l = []
for i in top_IDs_total.loc[flat_list,:].sort_values(by='Category').index:
    name = i
    for j in ['+']:
        l.append(['+','%s'%i,int(top_IDs_total.loc[i]['%s'%j])])
sankey = hv.Sankey(l)
sankey.opts(label_position='inner', width=320, height=500, cmap='tab20b'
            ,edge_line_width=0,label_text_font_size='13pt',node_line_width=0)

sankey = hv.Sankey(l,edge_line_width=0,label_text_font_size='12pt',node_line_width=0,label_position='inner', width=600, height=500)
export_svg(sankey, 'mdSAM_all_right2left.svg')


In [ ]:
l = [['+', 'Negative(DE)', 10],
 ['+', 'Positive (RKH)', 10],
 ['+', 'Aromatic (YWF)', 10],
 ['+', 'Hydrophobic (GAVILM)', 10],
 ['+', 'Polar(TSNQC)', 10],
 ['+', '$\sum\Delta G_{\beta}$', 10]]
sankey = hv.Sankey(l)
sankey.opts(label_position='inner', width=300, height=1000, cmap='tab20b'
            ,edge_line_width=0,label_text_font_size='14pt',node_line_width=0)
sankey = hv.Sankey(l,edge_line_width=0,label_text_font_size='12pt',node_line_width=0,label_position='inner', width=600, height=500)

export_svg(sankey, 'tsne_test.svg')


In [ ]:
l = []
for i in top_IDs_total.loc[top_IDs_df.sort_values(by='+').index].tail(11).index:
    name = i
    for j in ['+']:
        l.append(['%s'%i,'+',int(top_IDs_total.loc[i]['%s'%j])])
sankey = hv.Sankey(l)
sankey.opts(label_position='inner', width=250, height=500, cmap='tab20b'
            ,edge_line_width=0,label_text_font_size='13pt',node_line_width=0)
#export_svg(sankey, 'mdSAM_low.svg')
#sankey = hv.Sankey(l,edge_line_width=0,label_text_font_size='12pt',node_line_width=0,label_position='inner', width=600, height=500)
# sankey.opts(width=250, height=750)

#hv.save(sankey, 'penguin_plot.png', fmt='png')


In [ ]:
top_IDs_total.loc['Electron transport','+'] = 1 # pseudocount

In [ ]:
top_IDs_total.loc['Cell adhesion']#.values.sum()

In [ ]:
# l = []
for i in top_IDs_total.loc[top_IDs_df.sort_values(by='+').index].head(5).index:
    name = i
    for j in ['+']:
        l.append(['%s'%i,'+',int(top_IDs_total.loc[i]['%s'%j])])
sankey = hv.Sankey(l)
sankey.opts(label_position='inner', width=300, height=700, cmap='tab20b'
            ,edge_line_width=0,label_text_font_size='10pt',node_line_width=0)
#sankey = hv.Sankey(l,edge_line_width=0,label_text_font_size='12pt',node_line_width=0,label_position='inner', width=600, height=500)
#export_svg(sankey, 'mdSAM_all.svg')


In [ ]:
pd.DataFrame(l).to_pickle('Sankey_df.pkl')

In [ ]:
sankey = hv.Sankey(l)
sankey.opts(label_position='inner', width=300, height=650, cmap='tab20b'
            ,edge_line_width=0,label_text_font_size='12pt',node_line_width=0)
#export_svg(sankey, 'mdSAM_all.svg')

In [ ]:
import matplotlib.colors as colors

In [ ]:
l =[]
for i in Receptor[Receptor.counts_total>20].iloc[2:].index:
    ID = Receptor.loc[i,'ID']
    disrcption = Receptor.loc[i,'disrcption']
    in_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0675')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))])
    not_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0675')!=-1)&(~hTMs_uniprot.Entry.isin(mdSAM.acc))])
    total = not_mdSAM+in_mdSAM
    l.append({'disrcption':disrcption,'val':100*(not_mdSAM/total),'mdSAM':'-'})
    l.append({'disrcption':disrcption,'val':100*(in_mdSAM/total),'mdSAM':'+'})
Receptor_df = pd.DataFrame(l).pivot_table(index='disrcption',columns='mdSAM',values='val')

l =[]
for i in Receptor[Receptor.counts_total>20].iloc[2:].index:
    ID = Receptor.loc[i,'ID']
    disrcption = Receptor.loc[i,'disrcption']
    in_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0675')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))])
    not_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0675')!=-1)&(~hTMs_uniprot.Entry.isin(mdSAM.acc))])
    total = not_mdSAM+in_mdSAM
    l.append({'disrcption':disrcption,'val':not_mdSAM,'mdSAM':'-'})
    l.append({'disrcption':disrcption,'val':in_mdSAM,'mdSAM':'+'})
Receptor_df_total = pd.DataFrame(l).pivot_table(index='disrcption',columns='mdSAM',values='val')

In [ ]:
sns.clustermap(Receptor_df,metric='cosine'
              ,col_cluster=False,
              figsize=(1, 5),dendrogram_ratio=.1,cmap="PuOr",annot=Receptor_df_total,fmt='.3g')#norm=colors.SymLogNorm(vmin=0, vmax=Transport_df.max().max(),linthresh=1)

In [ ]:
fig, ax = plt.subplots(figsize=(1.5,6))
sns.heatmap(Receptor_df.sort_values(by='+'),cmap="RdGy",
            annot=Receptor_df_total.loc[Receptor_df.sort_values(by='+').index],fmt='.3g',square=False,cbar_kws={"shrink": .5})#norm=colors.SymLogNorm(vmin=0, vmax=Transport_df.max().max(),linthresh=1)

In [ ]:
Receptor[Receptor.counts_total.between(20,800)]

In [ ]:
l =[]
for i in Transport[Transport.counts_total>50].iloc[2:].index:
    ID = Transport.loc[i,'ID']
    disrcption = Transport.loc[i,'disrcption']
    in_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0813')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))])
    not_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0813')!=-1)&(~hTMs_uniprot.Entry.isin(mdSAM.acc))])
    total = in_mdSAM+not_mdSAM
    
    #hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find('KW-0297')!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0675')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))]
    l.append({'disrcption':disrcption,'val':100*(not_mdSAM/total),'mdSAM':'-'})
    l.append({'disrcption':disrcption,'val':100*(in_mdSAM/total),'mdSAM':'+'})
Transport_df = pd.DataFrame(l).pivot_table(index='disrcption',columns='mdSAM',values='val')
l =[]
for i in Transport[Transport.counts_total>50].iloc[2:].index:
    ID = Transport.loc[i,'ID']
    disrcption = Transport.loc[i,'disrcption']
    in_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0813')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))])
    not_mdSAM = len(hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(ID)!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0813')!=-1)&(~hTMs_uniprot.Entry.isin(mdSAM.acc))])
    total = in_mdSAM+not_mdSAM
    
    #hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find('KW-0297')!=-1)&(hTMs_uniprot['Keyword ID'].str.find('KW-0675')!=-1)&(hTMs_uniprot.Entry.isin(mdSAM.acc))]
    l.append({'disrcption':disrcption,'val':not_mdSAM,'mdSAM':'-'})
    l.append({'disrcption':disrcption,'val':in_mdSAM,'mdSAM':'+'})
Transport_df_values = pd.DataFrame(l).pivot_table(index='disrcption',columns='mdSAM',values='val')


In [ ]:
divnorm = colors.TwoSlopeNorm(vmin=0., vcenter=75, vmax=100)

In [ ]:
sns.clustermap(Transport_df,metric='cosine'
              ,col_cluster=False,
              figsize=(1, 5),dendrogram_ratio=.1,cmap="PuOr",annot=Transport_df_values,fmt='.3g')#norm=colors.SymLogNorm(vmin=0, vmax=Transport_df.max().max(),linthresh=1)

In [ ]:
fig, ax = plt.subplots(figsize=(1.5,6))
sns.heatmap(Transport_df.sort_values(by='+'),cmap=ww,
            annot=Transport_df_values.loc[Transport_df.sort_values(by='+').index],fmt='.3g',square=False,cbar_kws={"shrink": .5})#norm=colors.SymLogNorm(vmin=0, vmax=Transport_df.max().max(),linthresh=1)

In [ ]:
Transport_df

In [ ]:
iris = sns.load_dataset("iris")
#species = iris.pop("species")
iris

In [ ]:
nodes = ["PhD", "Career Outside Science",  "Early Career Researcher", "Research Staff",
         "Permanent Research Staff",  "Professor",  "Non-Academic Research"]
nodes = hv.Dataset(enumerate(nodes), 'index', 'label')
edges = [
    (0, 1, 53), (0, 2, 47), (2, 6, 17), (2, 3, 30), (3, 1, 22.5), (3, 4, 3.5), (3, 6, 4.), (4, 5, 0.45)   
]

value_dim = hv.Dimension('Percentage', unit='%')
careers = hv.Sankey((edges, nodes), ['From', 'To'], vdims=value_dim)

careers.opts(
    opts.Sankey(labels='label', label_position='right', width=900, height=300, cmap='Set1',
                edge_color=dim('To').str(), node_color=dim('index').str()))

In [ ]:
accs  = set(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(10,2000)].ID)].acc)
keys = list(set(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(10,2000)].ID)].ID))
len(accs),len(keys)


In [ ]:

l = []
for acc in tqdm.tqdm(hTMs_uniprot[hTMs_uniprot.Entry.isin(accs)].index):#(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    acc_name = hTMs_uniprot.loc[acc,'Entry']
    for keyword in keys:#.between(50,1323)
        if hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(keyword)!=-1)&(hTMs_uniprot.Entry==acc_name)].empty ==False:
            l.append({'acc':acc_name,'ID':keyword,'binary':1})
        else:
            l.append({'acc':acc_name,'ID':keyword,'binary':0})
x = pd.DataFrame(l).pivot_table(index='acc',columns='ID',values='binary')
x = x[x.index.isin(AF2[AF2.pLDDT!='A'].acc)]
x       

In [ ]:
reducer = umap.UMAP(random_state=42,n_neighbors=10)#,metric='manhattan')
embedding = reducer.fit_transform(x)
UMAP_plot_test = pd.DataFrame()
UMAP_plot_test['x'] =embedding[:,0]
UMAP_plot_test['y'] =embedding[:,1]
UMAP_plot_test['acc'] = x.index#UMAP_plot_test
for i in UMAP_plot_test.index:
    if UMAP_plot_test.loc[i,'acc'] in set(mdSAM.acc):
        UMAP_plot_test.loc[i,'mdSAM']=1
    else:
        UMAP_plot_test.loc[i,'mdSAM']=0
fig, ax = plt.subplots(figsize=(5,5))
sns.scatterplot(x='x',y='y',data=UMAP_plot_test[UMAP_plot_test.mdSAM!=2],s=10,hue='mdSAM',edgecolor=None,palette=['#B4B5B4','#C03830'])

In [ ]:
pca = PCA(n_components=2)
X_r = pca.fit(x).transform(x)
principalDf = pd.DataFrame(data = X_r
             , columns = ['x', 'y'])
principalDf['acc'] = x.index

In [ ]:
# principalComponents = TSNE(n_components=2, learning_rate='auto',init='random', perplexity=2).fit_transform(x)

# principalDf = pd.DataFrame(data = principalComponents
#              , columns = ['x', 'y'])
principalDf['acc'] = x.index#UMAP_plot_test
for i in principalDf.index:
    if principalDf.loc[i,'acc'] in set(mdSAM.acc):
        principalDf.loc[i,'mdSAM']=1
    else:
        principalDf.loc[i,'mdSAM']=0

fig, ax = plt.subplots(figsize=(5,5))

sns.scatterplot(x='x',y='y',data=principalDf,s=100,edgecolor=None,alpha=.1,hue='mdSAM')

In [ ]:
for j in tqdm.tqdm(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(10,1500)].ID)].drop_duplicates(subset='ID').index):
    key = KEYdf.loc[j,'ID']
    name = KEYdf.loc[j,'disrcption']
    for i in principalDf.index:
        acc = principalDf.loc[i,'acc']
        if hTMs_uniprot[(hTMs_uniprot.Entry==acc)&(hTMs_uniprot['Keyword ID'].str.find(key)!=-1)].empty:
            principalDf.loc[i,name] = 0
        else:
            principalDf.loc[i,name] = 1

In [ ]:
UMAP_plot_test      

In [ ]:
fig = plt.figure(figsize = (30,30),constrained_layout=False)
axs = fig.subplots(nrows=11, ncols=10,gridspec_kw = {'hspace':.5,'wspace':.4})
c = 0
for i in UMAP_plot_test.columns[4::]:
    ind = UMAP_plot_test[UMAP_plot_test['%s'%i]>0].index
    #fig, ax = plt.subplots(figsize=(3, 3))
    # sns.scatterplot(x='x',y='y',data=UMAP_plot_test.loc[ind],s=10,hue='%s'%'mdSAM',
    #                 alpha=1,ax=axs.flat[c],palette=['#B4B5B4','#C03830'],edgecolor='k')
    sns.scatterplot(x='x',y='y',data=UMAP_plot_test,s=1,hue='%s'%i,
                    ax=axs.flat[c],palette=['#B4B5B4','#C03830'],edgecolor=None,zorder=0)
    
    # sns.scatterplot(x='x',y='y',data=principalDf[(principalDf['%s'%i]==1)&(principalDf.mdSAM==1)],s=100,edgecolor='k'
    #                  ,color='#C03830',alpha=1,ax=axs.flat[c])
    axs.flat[c].legend([],frameon=False)
    axs.flat[c].set_title('%s'%i,fontsize=10)
    axs.flat[c].set_xlabel('')
    axs.flat[c].set_ylabel('')
    if c !=0:
        axs.flat[c].set_yticklabels([])
        axs.flat[c].set_xticklabels([])
    c = c+1

In [ ]:
fig = plt.figure(figsize = (30,30),constrained_layout=False)
axs = fig.subplots(nrows=11, ncols=10,gridspec_kw = {'hspace':.5,'wspace':.4})
c = 0
for i in principalDf.columns[4::]:
    ind = principalDf[principalDf['%s'%i]>0].index
    # #fig, ax = plt.subplots(figsize=(3, 3))
    # sns.scatterplot(x='x',y='y',data=principalDf.loc[ind],s=10,hue='%s'%'mdSAM',
    #                 alpha=1,ax=axs.flat[c],palette=['#B4B5B4','#C03830'],edgecolor='k')
    # sns.scatterplot(x='x',y='y',data=principalDf,s=1,
    #                 ax=axs.flat[c],color='yellow',edgecolor=None,alpha=0.05,zorder=0)
    sns.scatterplot(x='x',y='y',data=principalDf,s=1,hue='%s'%i,
                    ax=axs.flat[c],palette=['#B4B5B4','#C03830'],edgecolor=None,zorder=0)    
    # sns.scatterplot(x='x',y='y',data=principalDf[(principalDf['%s'%i]==1)&(principalDf.mdSAM==1)],s=100,edgecolor='k'
    #                  ,color='#C03830',alpha=1,ax=axs.flat[c])
    axs.flat[c].legend([],frameon=False)
    axs.flat[c].set_title('%s'%i,fontsize=10)
    axs.flat[c].set_xlabel('')
    axs.flat[c].set_ylabel('')
    if c !=0:
        axs.flat[c].set_yticklabels([])
        axs.flat[c].set_xticklabels([])
    c = c+1

In [ ]:
reducer = umap.UMAP(random_state=42,n_neighbors=2,metric='manhattan')
embedding = reducer.fit_transform(x)
UMAP_plot_test = pd.DataFrame()
UMAP_plot_test['x'] =embedding[:,0]
UMAP_plot_test['y'] =embedding[:,1]
UMAP_plot_test['acc'] = x.index#UMAP_plot_test
for i in UMAP_plot_test.index:
    if UMAP_plot_test.loc[i,'acc'] in set(mdSAM.acc):
        UMAP_plot_test.loc[i,'mdSAM']=1
    else:
        UMAP_plot_test.loc[i,'mdSAM']=0
fig, ax = plt.subplots(figsize=(5,5))
sns.scatterplot(x='x',y='y',data=UMAP_plot_test[UMAP_plot_test.mdSAM!=2],s=5,hue='mdSAM',edgecolor=None)

In [ ]:
KeyWord_df[KeyWord_df.Category=='Molecular function']


KEY = []
for j in hTMs_uniprot[(~pd.isna(hTMs_uniprot['Keyword ID']))].index: #(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    #hTMs_uniprot.loc[0]['Gene Ontology (biological process)']
    t= [i.replace(' ','') for i in hTMs_uniprot.loc[j,'Keyword ID'].split(';')]
    if len(set.intersection(set(KeyWord_df[KeyWord_df.Category=='Biological process']['Keyword ID']),set(t)))>0:
        for key in set.intersection(set(KeyWord_df[KeyWord_df.Category=='Biological process']['Keyword ID']),set(t)):
            KEY.append({'ID':key,'disrcption':KeyWord_df[KeyWord_df['Keyword ID']==key].Name.values[0],'acc':hTMs_uniprot.loc[j,'Entry']})
KEYdf_Biological = pd.DataFrame(KEY)
KEYdf_counts_Biological = KEYdf_Biological.ID.value_counts().reset_index()

KEYdf_counts_Biological = KEYdf_counts_Biological.rename(columns={'ID': 'counts_total','index':'ID'})

In [ ]:
for i in KEYdf_counts_Biological.index:
    KEYdf_counts_Biological.loc[i,'disrcption'] = KEYdf_Biological[KEYdf_Biological.ID==KEYdf_counts_Biological.loc[i,'ID']].iloc[0].disrcption

In [ ]:
KEYdf_counts_Biological[KEYdf_counts_Biological.counts_total.between(50,1200)]
accs_Biological  = set(KEYdf_Biological[KEYdf_Biological.ID.isin(KEYdf_Biological[KEYdf_Biological.counts_total.between(20,1500)].ID)].acc)
key_Biological = list(set(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(20,1500)].ID)].ID))


In [ ]:
KeyWord_df[KeyWord_df.Category=='Molecular function']


KEY = []
for j in hTMs_uniprot[(~pd.isna(hTMs_uniprot['Keyword ID']))].index: #(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    #hTMs_uniprot.loc[0]['Gene Ontology (biological process)']
    t= [i.replace(' ','') for i in hTMs_uniprot.loc[j,'Keyword ID'].split(';')]
    if len(set.intersection(set(KeyWord_df[KeyWord_df.Category=='Molecular function']['Keyword ID']),set(t)))>0:
        for key in set.intersection(set(KeyWord_df[KeyWord_df.Category=='Molecular function']['Keyword ID']),set(t)):
            KEY.append({'ID':key,'disrcption':KeyWord_df[KeyWord_df['Keyword ID']==key].Name.values[0],'acc':hTMs_uniprot.loc[j,'Entry']})
KEYdf = pd.DataFrame(KEY)
KEYdf_counts = KEYdf.ID.value_counts().reset_index()

KEYdf_counts= KEYdf_counts.rename(columns={'ID': 'counts_total','index':'ID'})

In [ ]:
for i in KEYdf_counts.index:
    KEYdf_counts.loc[i,'disrcption'] = KEYdf[KEYdf.ID==KEYdf_counts.loc[i,'ID']].iloc[0].disrcption

In [ ]:
KEYdf_counts.head(30)
accs  = set(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(20,1500)].ID)].acc)


In [ ]:
accs  = set(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(20,1500)].ID)].acc)
keys = list(set(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(20,1500)].ID)].ID))


In [ ]:
l = []
for acc in tqdm.tqdm(hTMs_uniprot[hTMs_uniprot.Entry.isin(accs)].index):#(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    acc_name = hTMs_uniprot.loc[acc,'Entry']
    for keyword in keys:#.between(50,1323)
        if hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(keyword)!=-1)&(hTMs_uniprot.Entry==acc_name)].empty ==False:
            l.append({'acc':acc_name,'ID':keyword,'binary':1})
        else:
            l.append({'acc':acc_name,'ID':keyword,'binary':0})
x = pd.DataFrame(l).pivot_table(index='acc',columns='ID',values='binary')
x = x[x.index.isin(AF2[AF2.pLDDT!='A'].acc)]
x       

In [ ]:
for j in tqdm.tqdm(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(20,1500)].ID)].drop_duplicates(subset='ID').index):
    key = KEYdf.loc[j,'ID']
    name = KEYdf.loc[j,'disrcption']
    for i in principalDf.index:
        acc = principalDf.loc[i,'acc']
        if hTMs_uniprot[(hTMs_uniprot.Entry==acc)&(hTMs_uniprot['Keyword ID'].str.find(key)!=-1)].empty:
            principalDf.loc[i,name] = 0
        else:
            principalDf.loc[i,name] = 1

In [ ]:
principalDf.columns

In [ ]:
sns.scatterplot(x='x',y='y',data=principalDf,s=3,edgecolor=None,hue='%s'%'Ion channel',
                    alpha=1,palette=['#B4B5B4','#C03830'])

In [ ]:
reducer = umap.UMAP(random_state=42,n_neighbors=2)#,metric='manhattan')
embedding = reducer.fit_transform(x)
UMAP_plot_test = pd.DataFrame()
UMAP_plot_test['x'] =embedding[:,0]
UMAP_plot_test['y'] =embedding[:,1]
UMAP_plot_test['acc'] = x.index#UMAP_plot_test
for i in UMAP_plot_test.index:
    if UMAP_plot_test.loc[i,'acc'] in set(mdSAM.acc):
        UMAP_plot_test.loc[i,'mdSAM']=1
    else:
        UMAP_plot_test.loc[i,'mdSAM']=0

In [ ]:
fig, ax = plt.subplots(figsize=(15,15))
sns.scatterplot(x='x',y='y',data=UMAP_plot_test[UMAP_plot_test.mdSAM!=2],s=5,hue='mdSAM',edgecolor=None)

In [ ]:
for j in tqdm.tqdm(KEYdf[KEYdf.ID.isin(KEYdf_counts[KEYdf_counts.counts_total.between(20,1500)].ID)].drop_duplicates(subset='ID').index):
    key = KEYdf.loc[j,'ID']
    name = KEYdf.loc[j,'disrcption']
    for i in UMAP_plot_test.index:
        acc = UMAP_plot_test.loc[i,'acc']
        if hTMs_uniprot[(hTMs_uniprot.Entry==acc)&(hTMs_uniprot['Keyword ID'].str.find(key)!=-1)].empty:
            UMAP_plot_test.loc[i,name] = 0
        else:
            UMAP_plot_test.loc[i,name] = 1

In [ ]:
sns.scatterplot(x='x',y='y',data=UMAP_plot_test,s=3,edgecolor=None,hue='%s'%'Translocase',
                    alpha=1,palette=['#B4B5B4','#C03830'])

In [ ]:
fig = plt.figure(figsize = (30,30),constrained_layout=False)
axs = fig.subplots(nrows=10, ncols=10,gridspec_kw = {'hspace':.5,'wspace':.5})
c = 0
for i in UMAP_plot_test.columns[4:4+100]:
    
    #fig, ax = plt.subplots(figsize=(3, 3))
    sns.scatterplot(x='x',y='y',data=UMAP_plot_test,s=3,edgecolor=None,hue='%s'%i,
                    alpha=1,ax=axs.flat[c],palette=['#B4B5B4','#C03830'])
    # sns.scatterplot(x='x',y='y',data=principalDf[(principalDf['%s'%i]==1)&(principalDf.mdSAM==1)],s=100,edgecolor='k'
    #                  ,color='#C03830',alpha=1,ax=axs.flat[c])
    axs.flat[c].legend([],frameon=False)
    axs.flat[c].set_title('%s'%i,fontsize=10)
    axs.flat[c].set_xlabel('')
    axs.flat[c].set_ylabel('')
    if c !=0:
        axs.flat[c].set_yticklabels([])
        axs.flat[c].set_xticklabels([])
    c = c+1

In [ ]:
GO = []
for j in hTMs_uniprot[(~pd.isna(hTMs_uniprot['Gene Ontology (molecular function)']))].index: #(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    #hTMs_uniprot.loc[0]['Gene Ontology (biological process)']
    t = {i.split('[')[1].split(']')[0]:i.split('[GO:')[0][:-1] for i in hTMs_uniprot.loc[j,'Gene Ontology (molecular function)'].split(';')}
    for key,value in t.items():
        if value[0] ==' ':
            GO.append({'GO':key,'disrcption':value[1:]})
        else:
            GO.append({'GO':key,'disrcption':value[:]})
GOdf = pd.DataFrame(GO)
GOdf = GOdf.value_counts().reset_index()
GOdf= GOdf.rename(columns={0: 'counts_in_mdSAMs'})

l = []
for acc in tqdm.tqdm(hTMs_uniprot[(~pd.isna(hTMs_uniprot['Gene Ontology (molecular function)']))].index):#(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    acc_name = hTMs_uniprot.loc[acc,'Entry']
    for keyword in GOdf[GOdf.counts_in_mdSAMs.between(10,700)].GO:#.between(50,1323)
        if hTMs_uniprot[(hTMs_uniprot['Gene Ontology (molecular function)'].str.find(keyword)!=-1)&(hTMs_uniprot.Entry==acc_name)].empty ==False:
            l.append({'acc':acc_name,'ID':keyword,'binary':1})
        else:
            l.append({'acc':acc_name,'ID':keyword,'binary':0})
x = pd.DataFrame(l).pivot_table(index='acc',columns='ID',values='binary')
x = x[x.index.isin(AF2[AF2.pLDDT!='A'].acc)]
x       

In [ ]:
sns.scatterplot(x='x',y='y',data=principalDf,s=3,edgecolor=None,hue='%s'%'protein homodimerization activity',
                    alpha=1,palette=['#B4B5B4','#C03830'])

In [ ]:
fig = plt.figure(figsize = (30,30),constrained_layout=False)
axs = fig.subplots(nrows=10, ncols=10,gridspec_kw = {'hspace':.5,'wspace':.5})
c = 0
for i in principalDf.columns[4:4+100]:
    
    #fig, ax = plt.subplots(figsize=(3, 3))
    sns.scatterplot(x='x',y='y',data=principalDf,s=3,edgecolor=None,hue='%s'%i,
                    alpha=1,ax=axs.flat[c],palette=['#B4B5B4','#C03830'])
    # sns.scatterplot(x='x',y='y',data=principalDf[(principalDf['%s'%i]==1)&(principalDf.mdSAM==1)],s=100,edgecolor='k'
    #                  ,color='#C03830',alpha=1,ax=axs.flat[c])
    axs.flat[c].legend([],frameon=False)
    axs.flat[c].set_title('%s'%i,fontsize=10)
    axs.flat[c].set_xlabel('')
    axs.flat[c].set_ylabel('')
    if c !=0:
        axs.flat[c].set_yticklabels([])
        axs.flat[c].set_xticklabels([])
    c = c+1

In [ ]:
l = []
for i in hTMs_uniprot[(hTMs_uniprot.Entry.isin(mdSAM.acc))&(~pd.isna(hTMs_uniprot['Gene Ontology (molecular function)']))].index:
    #l.append({i.split('[')[1].split(']')[0]:i.split('[')[0][:-1] for i in hTMs_uniprot.loc[i,'Gene Ontology (biological process)'].split(';')})
    l.append([i.split('[')[1].split(']')[0] for i in hTMs_uniprot.loc[i,'Gene Ontology (molecular function)'].split(';')])
#pd.Series([item for sublist in l for item in sublist]).value_counts()#

In [ ]:
c =[]
for i in pd.Series([item for sublist in l for item in sublist]).value_counts().reset_index()['index']:
    si = len(hTMs_uniprot[(hTMs_uniprot.Entry.isin(mdSAM.acc))&(hTMs_uniprot['Gene Ontology (molecular function)'].str.find(i)!=-1)&
(~pd.isna(hTMs_uniprot['Gene Ontology (molecular function)']))])
    c.append({'GO':i,'number':si})

In [ ]:
pd.DataFrame(c).reset_index()

In [ ]:
len(hTMs_uniprot[(hTMs_uniprot.Entry.isin(mdSAM.acc))&(hTMs_uniprot['Gene Ontology (molecular function)'].str.find('GO:0004930')!=-1)&
(~pd.isna(hTMs_uniprot['Gene Ontology (molecular function)']))])

In [ ]:
hTMs_uniprot.loc[4919,'Gene Ontology (molecular function)']

In [ ]:
l = []
for i in hTMs_uniprot[(hTMs_uniprot.Entry.isin(mdSAM.acc))&(~pd.isna(hTMs_uniprot['Gene Ontology (molecular function)']))].index:
    #l.append({i.split('[')[1].split(']')[0]:i.split('[')[0][:-1] for i in hTMs_uniprot.loc[i,'Gene Ontology (biological process)'].split(';')})
    l.append([i.split('[')[1].split(']')[0] for i in hTMs_uniprot.loc[i,'Gene Ontology (molecular function)'].split(';')])
#pd.Series([item for sublist in l for item in sublist]).value_counts()#

In [ ]:
[i.split('[')[1].split(']')[0] for i in hTMs_uniprot.loc[4919,'Gene Ontology (molecular function)'].split(';')]

In [ ]:
hTMs_uniprot.loc[4919,'Gene Ontology (biological process)']

In [ ]:
for j in [i.split('[')[1].split(']')[0] for i in hTMs_uniprot.loc[4919,'Gene Ontology (molecular function)'].split(';')]:
    print(len(hTMs_uniprot[(hTMs_uniprot.Entry.isin(mdSAM.acc))&(hTMs_uniprot['Gene Ontology (molecular function)'].str.find(j)!=-1)]))

In [ ]:
hTMs_uniprot.loc[8,'Gene Ontology (biological process)']

In [ ]:
hTMs_uniprot = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_10_27.tsv',delimiter='\t')
hTMs_uniprot = hTMs_uniprot[~pd.isna(hTMs_uniprot.Keywords)]
KeyWord_df = pd.read_csv('/Users/aelazar/Downloads/keywords_all_2023_10_27.tsv',delimiter='\t')

In [ ]:

l = [i.split(';') for i in hTMs_uniprot[hTMs_uniprot.Entry.isin(mdSAM.acc)]['Keyword ID'].values]
KeyWord_MP = pd.Series([item for sublist in l for item in sublist]).value_counts().reset_index()
KeyWord_MP= KeyWord_MP.rename(columns={0: 'counts','index':'ID'})
KeyWord_MP['ID'] = KeyWord_MP['ID'].str.replace(r' ', '', regex=False)
for i in KeyWord_MP.index:
   KeyWord_MP.loc[i,'name'] =  KeyWord_df[KeyWord_df['Keyword ID']==KeyWord_MP.loc[i,'ID']]['Name'].values[0]

In [ ]:
KeyWord_MP_MF = list((set.intersection(set(KeyWord_MP['ID'].to_list())
                 ,set(KeyWord_df[KeyWord_df.Category=='Molecular function']['Keyword ID'].to_list()))))

In [ ]:
KeyWord_df[KeyWord_df.Category=='Molecular function']['Keyword ID'].to_list()

In [ ]:
KeyWord_MP[KeyWord_MP.counts.between(1,4000)].to_csv('KeyWord_MP.csv')

In [ ]:
for i in hTMs_uniprot.index:
    hTMs_uniprot.loc[i,'No.Keywords']= len([j.split(';') for j in hTMs_uniprot.loc[i].Keywords])
    hTMs_uniprot.loc[i,'No.Keywords_IDs']= len([j.split(';') for j in hTMs_uniprot.loc[i]['Keyword ID']])

l = [i.split(';') for i in hTMs_uniprot['Keyword ID'].values]
KeyWord_MP = pd.Series([item for sublist in l for item in sublist]).value_counts().reset_index()
KeyWord_MP= KeyWord_MP.rename(columns={0: 'counts','index':'ID'})
KeyWord_MP['ID'] = KeyWord_MP['ID'].str.replace(r' ', '', regex=False)
for i in KeyWord_MP.index:
   KeyWord_MP.loc[i,'name'] =  KeyWord_df[KeyWord_df['Keyword ID']==KeyWord_MP.loc[i,'ID']]['Name'].values[0]
KeyWord_MP[KeyWord_MP.counts.between(1,4000)].to_csv('KeyWord_MP.csv')

In [ ]:
#key_subset =['KW-0675','KW-0813','KW-0297','KW-0479','KW-0406','KW-0808','KW-0393','KW-0106','KW-0407','KW-0443','KW-0130','KW-0966','KW-0378','KW-0547','KW-0770','KW-0449','KW-0391','KW-0067','KW-0552','KW-0217','KW-0851','KW-0915','KW-0523','KW-0769','KW-0653','KW-0630','KW-0739','KW-0965','KW-0633','KW-0053','KW-0399','KW-0444','KW-0445','KW-1064','KW-0418','KW-1278','KW-0109','KW-0863','KW-0631','KW-0050','KW-0524','KW-0107','KW-0446','KW-0206','KW-0482','KW-0829','KW-0965']
#key_subset = ['KW-0325','KW-1015','KW-0025','KW-0675','KW-0813','KW-0025','KW-0297','KW-0479','KW-0406','KW-0808','KW-0716','KW-0903','KW-0552','KW-0393','KW-0106','KW-0407','KW-0443','KW-0130','KW-0966','KW-0378','KW-0963','KW-0967','KW-0547','KW-0770','KW-0832','KW-0449','KW-0391','KW-0564','KW-0175','KW-0999','KW-0067','KW-0458','KW-0217','KW-0945','KW-0007','KW-0328','KW-0460','KW-0991','KW-0221','KW-0851','KW-0325','KW-0523','KW-0245','KW-0628','KW-0560','KW-0007','KW-0769','KW-0653','KW-0630','KW-0739','KW-0965','KW-0633','KW-0887','KW-0645','KW-0053','KW-0399','KW-0430','KW-0433','KW-0444','KW-0445','KW-1064','KW-0418','KW-0209','KW-1278','KW-0109','KW-0464','KW-0165','KW-1071','KW-0863','KW-0631','KW-0408','KW-0050','KW-0488','KW-1015','KW-0809','KW-1183','KW-0524','KW-1274','KW-0107','KW-0012','KW-0395','KW-0446','KW-0206','KW-0833','KW-0869','KW-0249','KW-0482','KW-0829','KW-0744','KW-0753','KW-0276','KW-0844','KW-0796','KW-0969','KW-0160','KW-1010','KW-0349','KW-0521','KW-0037','KW-0520','KW-0865','KW-0879','KW-0254','KW-1207','KW-0029','KW-1208','KW-0762','KW-0679','KW-0225','KW-0965','KW-0720','KW-0919','KW-0804','KW-0072','KW-0503','KW-0622','KW-0805','KW-0175','KW-0456','KW-0594','KW-0112','KW-0900','KW-0682','KW-0654','KW-0145','KW-0219','KW-0532','KW-0153','KW-0894','KW-1064','KW-0931','KW-0723','KW-0202','KW-0811','KW-0401','KW-0656','KW-0122','KW-0285','KW-0746','KW-0771','KW-0337','KW-0238','KW-0375','KW-0274','KW-0040','KW-0303','KW-0834','KW-0186','KW-0067','KW-0864','KW-0275','KW-0904','KW-0297','KW-0768','KW-0053','KW-0282','KW-0703','KW-0914','KW-0977','KW-0143','KW-0012','KW-0765','KW-1009','KW-0752','KW-0511','KW-0342','KW-0947','KW-0024','KW-0050','KW-1023','KW-0043','KW-0903','KW-1186','KW-0413','KW-0095','KW-0029','KW-0963','KW-0221','KW-0278','KW-0130','KW-0144','KW-0950','KW-0491','KW-0358','KW-1004','KW-0551','KW-0272','KW-1215','KW-0912','KW-0356','KW-0730','KW-0339','KW-0550','KW-0898','KW-0038','KW-0519','KW-0970','KW-0442','KW-1008','KW-0068','KW-1007','KW-0410','KW-0131','KW-0756','KW-0326','KW-0908','KW-0094','KW-0129','KW-0966','KW-0451','KW-0024','KW-0010','KW-0182','KW-1268','KW-0729','KW-1026','KW-1063','KW-0039','KW-0877','KW-0436','KW-0429','KW-0517','KW-0600','KW-0681','KW-0424','KW-0176','KW-0968','KW-0051','KW-1020','KW-0740','KW-0989','KW-0431','KW-0379','KW-0119']
key_subset= ['KW-0675','KW-0813','KW-0407','KW-0770','KW-0130','KW-0966','KW-0297','KW-0406','KW-0967','KW-0552','KW-0391','KW-0965']
key_subset = KeyWord_df[KeyWord_df.Category=='Molecular function']['Keyword ID'].to_list()#KeyWord_MP_MF

In [ ]:
len(key_subset)

In [ ]:
l = []
for acc in tqdm.tqdm(hTMs_uniprot[hTMs_uniprot.Entry.isin(mdSAM.acc)].index):
    acc_name = hTMs_uniprot.loc[acc,'Entry']
    for keyword in key_subset:#KeyWord_MP.drop_duplicates(subset=['ID'])[KeyWord_MP.ID.isin(key_subset)].ID:#.between(50,1323)
        if hTMs_uniprot[(hTMs_uniprot['Keyword ID'].str.find(keyword)!=-1)&(hTMs_uniprot.Entry==acc_name)].empty ==False:
            l.append({'acc':acc_name,'ID':keyword,'binary':1})
        else:
            l.append({'acc':acc_name,'ID':keyword,'binary':0})
x = pd.DataFrame(l).pivot_table(index='acc',columns='ID',values='binary')
x = x[x.index.isin(AF2[AF2.pLDDT!='A'].acc)]
x       

In [ ]:
reducer = umap.UMAP(random_state=42,n_neighbors=5)#,metric='manhattan')
embedding = reducer.fit_transform(x)
UMAP_plot_test = pd.DataFrame()
UMAP_plot_test['x'] =embedding[:,0]
UMAP_plot_test['y'] =embedding[:,1]
UMAP_plot_test['acc'] = x.index#UMAP_plot_test
for i in UMAP_plot_test.index:
    if UMAP_plot_test.loc[i,'acc'] in set(mdSAM.acc):
        UMAP_plot_test.loc[i,'mdSAM']=1
    else:
        UMAP_plot_test.loc[i,'mdSAM']=0



In [ ]:
reducer = umap.UMAP(random_state=42,n_neighbors=5)#,metric='manhattan')
embedding = reducer.fit_transform(x)
UMAP_plot_test = pd.DataFrame()
UMAP_plot_test['x'] =embedding[:,0]
UMAP_plot_test['y'] =embedding[:,1]
UMAP_plot_test['acc'] = x.index#UMAP_plot_test
for i in UMAP_plot_test.index:
    if UMAP_plot_test.loc[i,'acc'] in set(mdSAM.acc):
        UMAP_plot_test.loc[i,'mdSAM']=1
    else:
        UMAP_plot_test.loc[i,'mdSAM']=0


fig, ax = plt.subplots(figsize=(5,5))
sns.scatterplot(x='x',y='y',data=UMAP_plot_test[UMAP_plot_test.mdSAM!=2],s=1,hue='mdSAM',edgecolor=None)


In [ ]:
for j in tqdm.tqdm(KeyWord_MP.drop_duplicates(subset=['ID'])[KeyWord_MP.ID.isin(key_subset)].index):
    key = KeyWord_MP.loc[j,'ID']
    name = KeyWord_MP.loc[j,'name']
    for i in UMAP_plot_test.index:
        acc = UMAP_plot_test.loc[i,'acc']
        if hTMs_uniprot[(hTMs_uniprot.Entry==acc)&(hTMs_uniprot['Keyword ID'].str.find(key)!=-1)].empty:
            UMAP_plot_test.loc[i,name] = 0
        else:
            UMAP_plot_test.loc[i,name] = 1

In [ ]:
UMAP_plot_test

In [ ]:
sns.scatterplot(x='x',y='y',data=UMAP_plot_test,s=.5,hue='mdSAM',edgecolor=None)
# plt.xlim([-15,5])
# plt.ylim([-5,5])
# plt.ylim([-15,0])

In [ ]:
sns.kdeplot(x='x',y='y',data=UMAP_plot_test[UMAP_plot_test.mdSAM==0])


In [ ]:
principalDf

In [ ]:
pca = PCA(n_components=2)
X_r = pca.fit(x).transform(x)
principalDf = pd.DataFrame(data = X_r
             , columns = ['x', 'y'])
principalDf['acc'] = x.index
#fig, ax = plt.subplots(figsize=(10, 10))



# for i in principalDf.index:
#     if principalDf.loc[i,'acc'] in set(mdSAM.acc):
#         principalDf.loc[i,'mdSAM']=1
#     else:
#         principalDf.loc[i,'mdSAM']=0


In [ ]:
for j in tqdm.tqdm(KeyWord_MP.drop_duplicates(subset=['ID'])[KeyWord_MP.ID.isin(key_subset)].index):
    key = KeyWord_MP.loc[j,'ID']
    name = KeyWord_MP.loc[j,'name']
    for i in principalDf.index:
        acc = principalDf.loc[i,'acc']
        if hTMs_uniprot[(hTMs_uniprot.Entry==acc)&(hTMs_uniprot['Keyword ID'].str.find(key)!=-1)].empty:
            principalDf.loc[i,name] = 0
        else:
            principalDf.loc[i,name] = 1

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x='x',y='y',data=principalDf,s=5,edgecolor=None,)#hue='%s'%'mdSAM',alpha=1)

In [ ]:
principalDf.columns[4:]

In [ ]:
for i in principalDf.columns[4:]:
    
    fig, ax = plt.subplots(figsize=(3, 3))
    sns.scatterplot(x='x',y='y',data=principalDf,s=10,edgecolor=None,hue='%s'%i,alpha=.1)#[principalDf['Cytoskeleton']==1]
    plt.show()

In [ ]:
sns.scatterplot(x='x',y='y',data=principalDf[principalDf.mdSAM==0],s=10,edgecolor=None,color='#B4B5B4',alpha=.75)
sns.scatterplot(x='x',y='y',data=principalDf[principalDf.mdSAM==1],s=10,edgecolor=None,color='#C03830',alpha=.75)#'#C03830','#',

In [ ]:
x['KW-0021'].sum()

In [ ]:
fig, ax = plt.subplots(figsize=(25, 5))
sns.heatmap(x)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x='x',y='y',data=UMAP_plot_test[UMAP_plot_test.mdSAM==0].sample(969),s=1,color='k',edgecolor=None)
# plt.xlim([20,35])
# plt.ylim([10,30])
#fig, ax = plt.subplots(figsize=(3, 3))
sns.scatterplot(x='x',y='y',data=UMAP_plot_test[UMAP_plot_test.mdSAM==1],s=1,color='r',edgecolor=None)
# plt.xlim([20,35])
# plt.ylim([10,30])

In [ ]:
from sklearn.manifold import TSNE
principalComponents = TSNE(n_components=2, learning_rate='auto',init='random', perplexity=5,metric='manhattan').fit_transform(x)


In [ ]:
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['x', 'y'])
principalDf['acc'] = x.index#UMAP_plot_test
fig, ax = plt.subplots(figsize=(10, 10))



for i in principalDf.index:
    if principalDf.loc[i,'acc'] in set(mdSAM.acc):
        principalDf.loc[i,'mdSAM']=1
    else:
        principalDf.loc[i,'mdSAM']=0
sns.scatterplot(x='x',y='y',data=principalDf,s=10,edgecolor=None,hue='mdSAM',alpha=.5)

In [ ]:
x

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
sns.scatterplot(x='x',y='y',data=principalDf,s=30,edgecolor=None,hue='mdSAM',alpha=1)
# sns.scatterplot(x='x',y='y',data=principalDf[(principalDf.mdSAM==1)],s=10,edgecolor=None
#                  ,color='#C03830',alpha=.5)
# sns.scatterplot(x='x',y='y',data=principalDf[(principalDf.mdSAM==0)],s=10,edgecolor=None
#                  ,color='#B4B5B4',alpha=.5)

In [ ]:
for j in tqdm.tqdm(KeyWord_MP.drop_duplicates(subset=['ID'])[KeyWord_MP.ID.isin(key_subset)].index):
    key = KeyWord_MP.loc[j,'ID']
    name = KeyWord_MP.loc[j,'name']
    for i in principalDf.index:
        acc = principalDf.loc[i,'acc']
        if hTMs_uniprot[(hTMs_uniprot.Entry==acc)&(hTMs_uniprot['Keyword ID'].str.find(key)!=-1)].empty:
            principalDf.loc[i,name] = 0
        else:
            principalDf.loc[i,name] = 1

In [ ]:
x[x.index.isin(principalDf[(principalDf.x.between(0,20))&(principalDf.y.between(0,20))].acc)]

In [ ]:
fig = plt.figure(figsize = (10,20),constrained_layout=False)
axs = fig.subplots(nrows=6, ncols=2,gridspec_kw = {'hspace':.5,'wspace':.5})
c = 0
for i in principalDf.columns[4:4+100]:
    
    #fig, ax = plt.subplots(figsize=(3, 3))
    sns.scatterplot(x='x',y='y',data=principalDf,s=3,edgecolor=None,hue='%s'%i,
                    alpha=1,ax=axs.flat[c],palette=['#B4B5B4','#C03830'])
    # sns.scatterplot(x='x',y='y',data=principalDf[(principalDf['%s'%i]==1)&(principalDf.mdSAM==1)],s=100,edgecolor='k'
    #                  ,color='#C03830',alpha=1,ax=axs.flat[c])
    axs.flat[c].legend([],frameon=False)
    axs.flat[c].set_title('%s'%i,fontsize=10)
    axs.flat[c].set_xlabel('')
    axs.flat[c].set_ylabel('')
    if c !=0:
        axs.flat[c].set_yticklabels([])
        axs.flat[c].set_xticklabels([])
    c = c+1

In [ ]:
principalDf[(principalDf['%s'%i]==1)&(principalDf.mdSAM==1)]

In [ ]:
hTMs_uniprot

In [ ]:
non_OR_GPCRs = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Olfaction')==-1)&(hTMs_uniprot.Keywords.str.find('G-protein coupled receptor')!=-1)].Entry
Receptor = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Olfaction')==-1)&(hTMs_uniprot.Keywords.str.find('Receptor')!=-1)].Entry
Cell_junction = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Cell junction')!=-1)].Entry
Kinase = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Tyrosine-protein kinase')!=-1)].Entry
Transferase = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Transferase')!=-1)].Entry
Ion_channel = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Ion channel')!=-1)].Entry
Immunity = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Immunity')!=-1)].Entry
Synapse = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Synapse')!=-1)].Entry
Cell_adhesion = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Cell adhesion')!=-1)].Entry
Metalloprotease = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Metalloprotease')!=-1)].Entry
Cytoskeleton = hTMs_uniprot[(hTMs_uniprot.Keywords.str.find('Cytoskeleton')!=-1)].Entry

In [ ]:
fam = [non_OR_GPCRs,Cell_junction,Kinase,Ion_channel,Immunity,Synapse,
       Cell_adhesion,Receptor,Metalloprotease,Cytoskeleton]
fam_names = ['GPCRs','Cell junction','RTK','Ion channel','Immunity','Synapse',
             'Cell adhesion','Receptor','Metalloprotease','Cytoskeleton']

In [ ]:
l =[]
for family,name in zip(fam,fam_names):
    
    for acc in family:
        if pd.isna(hTMs_uniprot[hTMs_uniprot.Entry==acc]['Gene Names'].values[0])==False:
            gene_name = hTMs_uniprot[hTMs_uniprot.Entry==acc]['Gene Names'].values[0].split(' ')[0]
            if acc in mdSAM.acc.values:
                l.append({'acc':acc,'mdSAM':1,'KeyWord':name,'gene_name':gene_name,'temp':'A'})
            else:
                l.append({'acc':acc,'mdSAM':0,'KeyWord':name,'gene_name':gene_name,'temp':'A'})

In [ ]:
df = pd.DataFrame(l)
#df.value_counts(['KeyWord','mdSAM']).reset_index()

In [ ]:
fig, ax = plt.subplots(figsize=(2,5)) 
sns.barplot(y='KeyWord',x=0,data=df.value_counts(['KeyWord','mdSAM']).reset_index(),hue='mdSAM')
#plt.xscale('log')
ax.grid()

In [ ]:
df[df.KeyWord=='Immunity'].value_counts('mdSAM')

In [ ]:
fig, ax = plt.subplots(figsize=(2,25)) 
sns.heatmap(df[df.KeyWord=='non_OR_GPCRs'].pivot_table(index='gene_name',columns='KeyWord',values='mdSAM').sort_values(by='non_OR_GPCRs',ascending=False))

In [ ]:
sns.clustermap(pd.DataFrame(l).pivot_table(index='KeyWord',columns='gene_name',values='mdSAM'))

In [ ]:
x = pd.DataFrame(l).pivot_table(index='gene_name',columns='KeyWord',values='mdSAM')
x#sns.clustermap(x)
#sns.heatmap(x.sort_values(by='Metalloprotease'),square=True)

In [ ]:
import plotly.express as px
data = dict(
    character=["Eve", "Cain", "Seth", "Enos", "Noam", "Abel", "Awan", "Enoch", "Azura"],
    parent=["", "Eve", "Eve", "Seth", "Seth", "Eve", "Eve", "Awan", "Eve" ],
    value=[10, 14, 12, 10, 2, 6, 6, 4, 4])

fig = px.sunburst(
    data,
    names='character',
    parents='parent',
    values='value',
)
fig.show()

In [ ]:
dir(go)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections import register_projection
from matplotlib.projections.polar import PolarAxes
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D


def radar_factory(num_vars, frame='circle'):
    """
    Create a radar chart with `num_vars` axes.

    This function creates a RadarAxes projection and registers it.

    Parameters
    ----------
    num_vars : int
        Number of variables for radar chart.
    frame : {'circle', 'polygon'}
        Shape of frame surrounding axes.

    """
    # calculate evenly-spaced axis angles
    theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)

    class RadarTransform(PolarAxes.PolarTransform):

        def transform_path_non_affine(self, path):
            # Paths with non-unit interpolation steps correspond to gridlines,
            # in which case we force interpolation (to defeat PolarTransform's
            # autoconversion to circular arcs).
            if path._interpolation_steps > 1:
                path = path.interpolated(num_vars)
            return Path(self.transform(path.vertices), path.codes)

    class RadarAxes(PolarAxes):

        name = 'radar'
        PolarTransform = RadarTransform

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            # rotate plot such that the first axis is at the top
            self.set_theta_zero_location('N')

        def fill(self, *args, closed=True, **kwargs):
            """Override fill so that line is closed by default"""
            return super().fill(closed=closed, *args, **kwargs)

        def plot(self, *args, **kwargs):
            """Override plot so that line is closed by default"""
            lines = super().plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)

        def _close_line(self, line):
            x, y = line.get_data()
            # FIXME: markers at x[0], y[0] get doubled-up
            if x[0] != x[-1]:
                x = np.append(x, x[0])
                y = np.append(y, y[0])
                line.set_data(x, y)

        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)

        def _gen_axes_patch(self):
            # The Axes patch must be centered at (0.5, 0.5) and of radius 0.5
            # in axes coordinates.
            if frame == 'circle':
                return Circle((0.5, 0.5), 0.5)
            elif frame == 'polygon':
                return RegularPolygon((0.5, 0.5), num_vars,
                                      radius=.5, edgecolor="k")
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

        def _gen_axes_spines(self):
            if frame == 'circle':
                return super()._gen_axes_spines()
            elif frame == 'polygon':
                # spine_type must be 'left'/'right'/'top'/'bottom'/'circle'.
                spine = Spine(axes=self,
                              spine_type='circle',
                              path=Path.unit_regular_polygon(num_vars))
                # unit_regular_polygon gives a polygon of radius 1 centered at
                # (0, 0) but we want a polygon of radius 0.5 centered at (0.5,
                # 0.5) in axes coordinates.
                spine.set_transform(Affine2D().scale(.5).translate(.5, .5)
                                    + self.transAxes)
                return {'polar': spine}
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

    register_projection(RadarAxes)
    return theta


def example_data():
    # The following data is from the Denver Aerosol Sources and Health study.
    # See doi:10.1016/j.atmosenv.2008.12.017
    #
    # The data are pollution source profile estimates for five modeled
    # pollution sources (e.g., cars, wood-burning, etc) that emit 7-9 chemical
    # species. The radar charts are experimented with here to see if we can
    # nicely visualize how the modeled source profiles change across four
    # scenarios:
    #  1) No gas-phase species present, just seven particulate counts on
    #     Sulfate
    #     Nitrate
    #     Elemental Carbon (EC)
    #     Organic Carbon fraction 1 (OC)
    #     Organic Carbon fraction 2 (OC2)
    #     Organic Carbon fraction 3 (OC3)
    #     Pyrolyzed Organic Carbon (OP)
    #  2)Inclusion of gas-phase specie carbon monoxide (CO)
    #  3)Inclusion of gas-phase specie ozone (O3).
    #  4)Inclusion of both gas-phase species is present...
    data = [
        ['Sulfate', 'Nitrate', 'EC', 'OC1', 'OC2', 'OC3', 'OP', 'CO', 'O3'],
        ('Basecase', [
            [0.88, 0.01, 0.03, 0.03, 0.00, 0.06, 0.01, 0.00, 0.00],
            [0.07, 0.95, 0.04, 0.05, 0.00, 0.02, 0.01, 0.00, 0.00],
            [0.01, 0.02, 0.85, 0.19, 0.05, 0.10, 0.00, 0.00, 0.00],
            [0.02, 0.01, 0.07, 0.01, 0.21, 0.12, 0.98, 0.00, 0.00],
            [0.01, 0.01, 0.02, 0.71, 0.74, 0.70, 0.00, 0.00, 0.00]]),
        ('With CO', [
            [0.88, 0.02, 0.02, 0.02, 0.00, 0.05, 0.00, 0.05, 0.00],
            [0.08, 0.94, 0.04, 0.02, 0.00, 0.01, 0.12, 0.04, 0.00],
            [0.01, 0.01, 0.79, 0.10, 0.00, 0.05, 0.00, 0.31, 0.00],
            [0.00, 0.02, 0.03, 0.38, 0.31, 0.31, 0.00, 0.59, 0.00],
            [0.02, 0.02, 0.11, 0.47, 0.69, 0.58, 0.88, 0.00, 0.00]]),
        ('With O3', [
            [0.89, 0.01, 0.07, 0.00, 0.00, 0.05, 0.00, 0.00, 0.03],
            [0.07, 0.95, 0.05, 0.04, 0.00, 0.02, 0.12, 0.00, 0.00],
            [0.01, 0.02, 0.86, 0.27, 0.16, 0.19, 0.00, 0.00, 0.00],
            [0.01, 0.03, 0.00, 0.32, 0.29, 0.27, 0.00, 0.00, 0.95],
            [0.02, 0.00, 0.03, 0.37, 0.56, 0.47, 0.87, 0.00, 0.00]]),
        ('CO & O3', [
            [0.87, 0.01, 0.08, 0.00, 0.00, 0.04, 0.00, 0.00, 0.01],
            [0.09, 0.95, 0.02, 0.03, 0.00, 0.01, 0.13, 0.06, 0.00],
            [0.01, 0.02, 0.71, 0.24, 0.13, 0.16, 0.00, 0.50, 0.00],
            [0.01, 0.03, 0.00, 0.28, 0.24, 0.23, 0.00, 0.44, 0.88],
            [0.02, 0.00, 0.18, 0.45, 0.64, 0.55, 0.86, 0.00, 0.16]])
    ]
    return data


if __name__ == '__main__':
    N = 9
    theta = radar_factory(N, frame='polygon')

    data = example_data()
    spoke_labels = data.pop(0)

    fig, axs = plt.subplots(figsize=(9, 9), nrows=2, ncols=2,
                            subplot_kw=dict(projection='radar'))
    fig.subplots_adjust(wspace=0.25, hspace=0.20, top=0.85, bottom=0.05)

    colors = ['b', 'r', 'g', 'm', 'y']
    # Plot the four cases from the example data on separate axes
    for ax, (title, case_data) in zip(axs.flat, data):
        ax.set_rgrids([0.2, 0.4, 0.6, 0.8])
        ax.set_title(title, weight='bold', size='medium', position=(0.5, 1.1),
                     horizontalalignment='center', verticalalignment='center')
        for d, color in zip(case_data, colors):
            ax.plot(theta, d, color=color)
            ax.fill(theta, d, facecolor=color, alpha=0.25, label='_nolegend_')
        ax.set_varlabels(spoke_labels)

    # add legend relative to top-left plot
    labels = ('Factor 1', 'Factor 2', 'Factor 3', 'Factor 4', 'Factor 5')
    legend = axs[0, 0].legend(labels, loc=(0.9, .95),
                              labelspacing=0.1, fontsize='small')

    fig.text(0.5, 0.965, '5-Factor Solution Profiles Across Four Scenarios',
             horizontalalignment='center', color='black', weight='bold',
             size='large')


In [ ]:
N = len(ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name']).mean().reset_index()['ThT'].values)
v  = ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name']).mean().reset_index().sort_values(by='ThT')['ThT']
m = ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name']).mean() - ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name']).sem()
p = ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name']).mean() + ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name']).sem()
 
theta = radar_factory(N, frame='polygon')
fig, ax = plt.subplots(figsize=(6,6),subplot_kw=dict(projection='radar'))
ax.plot(theta, ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name','comb_new']).mean().reset_index().sort_values(by='ThT')['ThT'], color='#8C6785',lw=4,zorder=0)
# ax.plot(theta,m['ThT'].values, color='#8C6785',alpha=0.5)
# ax.plot(theta,p['ThT'].values, color='#8C6785',alpha=0.5)

ax.set_yscale('log')
ax.set_ylim([10,5e4])
ax.plot(np.linspace(0, 2*np.pi, 100), [200] * len(np.linspace(0, 2*np.pi, 100)), c='#D98883',zorder=0,lw=2,ls='--')
ax.plot(np.linspace(0, 2*np.pi, 100), [901.3333333333334] * len(np.linspace(0, 2*np.pi, 100)), c='#B8B8B8',lw=2,ls='--',zorder=0)
ax.set_varlabels(ThT_96[(ThT_96.gmm_new==1)].groupby(['Seq','gene_name','comb']).mean().reset_index().sort_values(by='ThT')['comb'].values)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.savefig('ThT_gmm_1_ploar_test.svg',bbox_inches='tight',transparent=False)

In [ ]:
for i in ThT_96[~pd.isna(ThT_96.comb)].index:
    ThT_96.loc[i,'comb_new'] = ThT_96.loc[i,'comb'].replace(' ','\t')

In [ ]:
ThT_96

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Generate some data
theta = np.linspace(0, 2*np.pi, 100)
r = np.sin(2*theta)

# Create a polar plot
plt.figure()
ax = plt.subplot(111, projection='polar')

# Plot the line


# Add labels, title, etc.
plt.title('Polar Plot with a Line')
plt.legend()

# Show the plot
plt.show()